# Hydrostratigraphic groups, correlation and sections

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Groups the accepted lithology intervals into hydrostratigraphic groups, screens borehole pairs for possible correlation (Steps 7C–7G; all decisions stay manual) and draws preliminary elevation-based sections (Step 7H).

| | |
|---|---|
| **Reads** | `Alwar_Accepted_Lithology_Intervals.csv`, `Alwar_Lithology_Classified_Intervals_Final.csv`, `Step6F_Borehole_Lithology_Elevation_Database.csv`, `Step6B_Borehole_Coordinates_For_DEM.csv` |
| **Writes** | `Alwar_Hydrostratigraphic_Intervals.csv`, `Step7C_…` to `Step7H2_…` registers, PDFs and PNGs under `data/BDR/Alwar/` |
| **Cells** | 19 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. Hydrostratigraphic groups
- 2. Correlation candidates (Steps 7C–7G)
- 3. Preliminary sections (Step 7H)

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["openpyxl"], modflow=False)
del TOKEN, URL


## 1. Hydrostratigraphic groups

**[orig #278]** **MAIN** Hydrostratigraphic intervals and group summary  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

# ---------------------------------------------------------
# 1. PATHS
# ---------------------------------------------------------
BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/Alwar_BDR_Model_Ready"
)
MODEL = BASE / "MODFLOW_Model_Input"

CLASS_FILE = MODEL / "Alwar_Lithology_Classified_Intervals_Final.csv"
ACCEPTED_FILE = MODEL / "Alwar_Accepted_Lithology_Intervals.csv"

OUT_INTERVALS = MODEL / "Alwar_Hydrostratigraphic_Intervals.csv"
OUT_GROUP_SUMMARY = MODEL / "Alwar_Hydrostratigraphic_Group_Summary.csv"
OUT_BOREHOLE_SUMMARY = MODEL / "Alwar_Borehole_Hydrostratigraphic_Summary.csv"
OUT_REVIEW = MODEL / "Alwar_Hydrostratigraphic_Review.csv"

# ---------------------------------------------------------
# 2. LOAD AND CHECK DATA
# ---------------------------------------------------------
classified = pd.read_csv(CLASS_FILE)
accepted = pd.read_csv(ACCEPTED_FILE)

KEYS = ["Borehole_ID", "Source_Row"]

required_class_cols = KEYS + [
    "Lithology_Class",
    "Classification_Confidence",
    "Classification_Method"
]

missing = [c for c in required_class_cols if c not in classified.columns]
if missing:
    raise ValueError(f"Missing classification columns: {missing}")

if accepted.duplicated(KEYS).any():
    raise ValueError("Duplicate Borehole_ID + Source_Row keys in accepted dataset.")

if classified.duplicated(KEYS).any():
    raise ValueError("Duplicate Borehole_ID + Source_Row keys in classified dataset.")

# Check that both files contain exactly the same interval keys
key_check = accepted[KEYS].merge(
    classified[KEYS],
    on=KEYS,
    how="outer",
    indicator=True
)

if not (key_check["_merge"] == "both").all():
    raise ValueError(
        "Accepted and classified datasets do not have identical interval keys. "
        "Inspect key_check before proceeding."
    )

# Compare source values before joining
source_check_cols = [
    "Source_From_m",
    "Source_To_m",
    "Source_Lithology"
]

for col in source_check_cols:
    if col in accepted.columns and col in classified.columns:
        a = accepted[col].fillna("__MISSING__").astype(str)
        b = classified[col].fillna("__MISSING__").astype(str)

        if not a.equals(b):
            raise ValueError(f"Source mismatch detected in {col}")

# Add classification fields to accepted intervals
classification_fields = KEYS + [
    "Lithology_Class",
    "Classification_Confidence",
    "Classification_Method"
]

df = accepted.merge(
    classified[classification_fields],
    on=KEYS,
    how="left",
    validate="one_to_one"
)

print("Accepted intervals:", len(accepted))
print("Classified intervals:", len(classified))
print("Joined intervals:", len(df))
print("Unique boreholes:", df["Borehole_ID"].nunique())

# ---------------------------------------------------------
# 3. CONSERVATIVE HYDROSTRATIGRAPHIC GROUPING
# ---------------------------------------------------------
def normalize_class(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = value.replace("–", "-").replace("—", "-")
    value = re.sub(r"\s+", " ", value)
    return value

# Map descriptive lithology classes to preliminary groups.
# Soil is deliberately kept separate.
GROUP_MAP = {
    # HSG-1: granular deposits
    "sand": ("HSG-1", "Granular deposits"),
    "fine sand": ("HSG-1", "Granular deposits"),
    "sand - grain size unspecified": ("HSG-1", "Granular deposits"),
    "mixed sand-fines": ("HSG-1", "Granular deposits"),
    "mixed sand-kankar": ("HSG-1", "Granular deposits"),

    # HSG-2: explicitly fine-grained descriptions
    "silt": ("HSG-2", "Fine-grained deposits"),
    "clay/silty material": ("HSG-2", "Fine-grained deposits"),

    # HSG-3: weathered material
    "weathered rock": ("HSG-3", "Weathered rock"),

    # HSG-4: consolidated lithologies
    "shale": ("HSG-4", "Consolidated bedrock"),
    "slate": ("HSG-4", "Consolidated bedrock"),
    "schist": ("HSG-4", "Consolidated bedrock"),
    "phyllite": ("HSG-4", "Consolidated bedrock"),
    "hard/consolidated rock": ("HSG-4", "Consolidated bedrock"),

    # HSG-5: insufficient grain-size information
    "soil": ("HSG-5", "Unspecified soil material"),
}

def assign_group(lith_class):
    key = normalize_class(lith_class)

    if key in GROUP_MAP:
        return GROUP_MAP[key]

    return ("UNASSIGNED", "Requires classification review")

group_results = df["Lithology_Class"].apply(assign_group)

df["Hydrostratigraphic_Group_ID"] = group_results.apply(lambda x: x[0])
df["Hydrostratigraphic_Group"] = group_results.apply(lambda x: x[1])

# ---------------------------------------------------------
# 4. ADD INTERPRETATION AND REVIEW FLAGS
# ---------------------------------------------------------
GROUP_DESCRIPTIONS = {
    "HSG-1": (
        "Sand-dominated or mixed granular deposits; "
        "potentially permeable, but hydraulic properties are unverified."
    ),
    "HSG-2": (
        "Explicitly described silt or clay/silty material; "
        "may have lower permeability, subject to field and test confirmation."
    ),
    "HSG-3": (
        "Weathered rock; hydraulic behaviour depends on weathering "
        "intensity, fractures and connected secondary porosity."
    ),
    "HSG-4": (
        "Consolidated lithologies; groundwater occurrence may depend "
        "on fractures, joints, bedding and weathering."
    ),
    "HSG-5": (
        "Soil description without sufficient grain-size information "
        "for a more specific lithological grouping."
    ),
    "UNASSIGNED": "Lithology class requires review before grouping."
}

df["Hydrostratigraphic_Description"] = (
    df["Hydrostratigraphic_Group_ID"]
    .map(GROUP_DESCRIPTIONS)
)

df["Grouping_Status"] = np.where(
    df["Hydrostratigraphic_Group_ID"] == "UNASSIGNED",
    "REVIEW_REQUIRED",
    "PRELIMINARY_LITHOLOGY_GROUP"
)

df["Hydraulic_Property_Status"] = (
    "NOT_ASSIGNED_NO_HYDRAULIC_TEST_DATA"
)

# Mixed classes are explicitly flagged
df["Mixed_Lithology_Flag"] = df["Lithology_Class"].apply(
    lambda x: (
        "YES" if normalize_class(x).startswith("mixed ")
        else "NO"
    )
)

# Flag intervals with low/uncertain classification confidence
df["Classification_Review_Flag"] = df[
    "Classification_Confidence"
].astype(str).str.strip().str.lower().apply(
    lambda x: "REVIEW" if x in ["low", "uncertain", "unknown", "nan", ""] else "NO"
)

# ---------------------------------------------------------
# 5. INTERVAL-LEVEL QUALITY CHECKS
# ---------------------------------------------------------
df["Depth_Interval_Valid"] = (
    pd.to_numeric(df["Source_From_m"], errors="coerce").notna()
    & pd.to_numeric(df["Source_To_m"], errors="coerce").notna()
    & (
        pd.to_numeric(df["Source_To_m"], errors="coerce")
        > pd.to_numeric(df["Source_From_m"], errors="coerce")
    )
)

df["Source_Interval_Thickness_m"] = (
    pd.to_numeric(df["Source_To_m"], errors="coerce")
    - pd.to_numeric(df["Source_From_m"], errors="coerce")
)

# Keep source-reported thickness separately; do not overwrite it.
df["Thickness_Comparison_Status"] = np.where(
    np.isclose(
        pd.to_numeric(df["Source_Thickness_m"], errors="coerce"),
        df["Source_Interval_Thickness_m"],
        equal_nan=False
    ),
    "MATCH",
    "CHECK_SOURCE_THICKNESS"
)

# ---------------------------------------------------------
# 6. GROUP SUMMARY
# ---------------------------------------------------------
group_summary = (
    df.groupby(
        ["Hydrostratigraphic_Group_ID", "Hydrostratigraphic_Group"],
        dropna=False
    )
    .agg(
        Interval_Count=("Borehole_ID", "size"),
        Borehole_Count=("Borehole_ID", "nunique"),
        Sum_Source_Reported_Thickness_m=("Source_Thickness_m", "sum"),
        Sum_Depth_Difference_m=("Source_Interval_Thickness_m", "sum"),
        Mixed_Interval_Count=("Mixed_Lithology_Flag",
                              lambda x: (x == "YES").sum()),
        Review_Interval_Count=("Classification_Review_Flag",
                              lambda x: (x == "REVIEW").sum())
    )
    .reset_index()
)

group_summary["Interpretation_Note"] = (
    "Descriptive interval totals only; not district-wide aquifer thickness."
)

# ---------------------------------------------------------
# 7. BOREHOLE SUMMARY
# ---------------------------------------------------------
df_sorted = df.sort_values(
    ["Borehole_ID", "Source_From_m", "Source_To_m"]
).copy()

def sequence_text(group):
    entries = []
    for _, row in group.iterrows():
        entries.append(
            f'{row["Hydrostratigraphic_Group_ID"]} '
            f'({row["Source_From_m"]}-{row["Source_To_m"]} m)'
        )
    return " | ".join(entries)

borehole_summary = (
    df_sorted.groupby("Borehole_ID")
    .agg(
        Dataset=("Dataset", "first"),
        Interval_Count=("Source_Row", "count"),
        First_Recorded_Depth_m=("Source_From_m", "min"),
        Maximum_Recorded_Depth_m=("Source_To_m", "max"),
        Hydrostratigraphic_Group_Count=(
            "Hydrostratigraphic_Group_ID", "nunique"
        ),
        Hydrostratigraphic_Sequence=(
            "Hydrostratigraphic_Group_ID", "size"
        )
    )
    .reset_index()
)

sequence_lookup = (
    df_sorted.groupby("Borehole_ID")
    .apply(sequence_text, include_groups=False)
    .rename("Ordered_Hydrostratigraphic_Sequence")
    .reset_index()
)

borehole_summary = borehole_summary.drop(
    columns=["Hydrostratigraphic_Sequence"]
).merge(sequence_lookup, on="Borehole_ID", how="left")

# Add group-specific interval counts per borehole
group_counts = pd.crosstab(
    df["Borehole_ID"],
    df["Hydrostratigraphic_Group_ID"]
).reset_index()

borehole_summary = borehole_summary.merge(
    group_counts, on="Borehole_ID", how="left"
)

# ---------------------------------------------------------
# 8. REVIEW DATASET
# ---------------------------------------------------------
review = df[
    (df["Grouping_Status"] == "REVIEW_REQUIRED")
    | (df["Classification_Review_Flag"] == "REVIEW")
    | (~df["Depth_Interval_Valid"])
    | (df["Thickness_Comparison_Status"] != "MATCH")
].copy()

# ---------------------------------------------------------
# 9. SAVE OUTPUTS
# ---------------------------------------------------------
df.to_csv(OUT_INTERVALS, index=False)
group_summary.to_csv(OUT_GROUP_SUMMARY, index=False)
borehole_summary.to_csv(OUT_BOREHOLE_SUMMARY, index=False)
review.to_csv(OUT_REVIEW, index=False)

# ---------------------------------------------------------
# 10. REPORT
# ---------------------------------------------------------
print("\n========== STEP 7B RESULTS ==========")
print("Hydrostratigraphic intervals:", len(df))
print("Unique boreholes:", df["Borehole_ID"].nunique())

print("\nGroup summary:")
display(group_summary)

print("\nBorehole summary:")
display(borehole_summary.head(10))

print("\nReview records:", len(review))
if len(review):
    display(
        review[
            [
                "Borehole_ID",
                "Source_Row",
                "Source_Lithology",
                "Lithology_Class",
                "Hydrostratigraphic_Group_ID",
                "Grouping_Status",
                "Thickness_Comparison_Status"
            ]
        ].head(20)
    )

print("\nSaved files:")
for p in [
    OUT_INTERVALS,
    OUT_GROUP_SUMMARY,
    OUT_BOREHOLE_SUMMARY,
    OUT_REVIEW
]:
    print(p)

**[orig #279]** Revised grouping (normalised lithology text); rebuild summaries  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

MODEL = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input"
)

FILE = MODEL / "Alwar_Hydrostratigraphic_Intervals.csv"

df = pd.read_csv(FILE)

# Normalize case, dashes, whitespace and spaces around slash
def normalize_class(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = value.replace("–", "-").replace("—", "-")
    value = re.sub(r"\s*/\s*", "/", value)
    value = re.sub(r"\s+", " ", value)
    return value

GROUP_MAP = {
    "sand": ("HSG-1", "Granular deposits"),
    "fine sand": ("HSG-1", "Granular deposits"),
    "sand - grain size unspecified": ("HSG-1", "Granular deposits"),
    "mixed sand-fines": ("HSG-1", "Granular deposits"),
    "mixed sand-kankar": ("HSG-1", "Granular deposits"),

    "silt": ("HSG-2", "Fine-grained deposits"),
    "clay/silty material": ("HSG-2", "Fine-grained deposits"),

    "weathered rock": ("HSG-3", "Weathered rock"),

    "shale": ("HSG-4", "Consolidated bedrock"),
    "slate": ("HSG-4", "Consolidated bedrock"),
    "schist": ("HSG-4", "Consolidated bedrock"),
    "phyllite": ("HSG-4", "Consolidated bedrock"),
    "hard/consolidated rock": ("HSG-4", "Consolidated bedrock"),

    "soil": ("HSG-5", "Unspecified soil material")
}

def assign_group(value):
    return GROUP_MAP.get(
        normalize_class(value),
        ("UNASSIGNED", "Requires classification review")
    )

mapped = df["Lithology_Class"].apply(assign_group)

df["Hydrostratigraphic_Group_ID"] = mapped.apply(lambda x: x[0])
df["Hydrostratigraphic_Group"] = mapped.apply(lambda x: x[1])

df["Grouping_Status"] = np.where(
    df["Hydrostratigraphic_Group_ID"] == "UNASSIGNED",
    "REVIEW_REQUIRED",
    "PRELIMINARY_LITHOLOGY_GROUP"
)

df["Hydraulic_Property_Status"] = (
    "NOT_ASSIGNED_NO_HYDRAULIC_TEST_DATA"
)

# Rebuild group summary
group_summary = (
    df.groupby(
        ["Hydrostratigraphic_Group_ID", "Hydrostratigraphic_Group"],
        dropna=False
    )
    .agg(
        Interval_Count=("Borehole_ID", "size"),
        Borehole_Count=("Borehole_ID", "nunique"),
        Sum_Source_Reported_Thickness_m=("Source_Thickness_m", "sum"),
        Sum_Depth_Difference_m=("Source_Interval_Thickness_m", "sum"),
        Mixed_Interval_Count=("Mixed_Lithology_Flag",
                              lambda x: (x == "YES").sum())
    )
    .reset_index()
)

group_summary["Interpretation_Note"] = (
    "Descriptive interval totals only; not district-wide aquifer thickness."
)

# Rebuild borehole summary
df_sorted = df.sort_values(
    ["Borehole_ID", "Source_From_m", "Source_To_m"]
)

def sequence_text(g):
    return " | ".join(
        f'{r.Hydrostratigraphic_Group_ID} '
        f'({r.Source_From_m}-{r.Source_To_m} m)'
        for r in g.itertuples()
    )

borehole_summary = (
    df_sorted.groupby("Borehole_ID")
    .agg(
        Dataset=("Dataset", "first"),
        Interval_Count=("Source_Row", "count"),
        First_Recorded_Depth_m=("Source_From_m", "min"),
        Maximum_Recorded_Depth_m=("Source_To_m", "max"),
        Hydrostratigraphic_Group_Count=(
            "Hydrostratigraphic_Group_ID", "nunique"
        )
    )
    .reset_index()
)

sequences = (
    df_sorted.groupby("Borehole_ID")
    .apply(sequence_text, include_groups=False)
    .rename("Ordered_Hydrostratigraphic_Sequence")
    .reset_index()
)

borehole_summary = borehole_summary.merge(
    sequences, on="Borehole_ID", how="left"
)

group_counts = pd.crosstab(
    df["Borehole_ID"],
    df["Hydrostratigraphic_Group_ID"]
).reset_index()

borehole_summary = borehole_summary.merge(
    group_counts, on="Borehole_ID", how="left"
)

# Review all unassigned intervals and thickness mismatches
review = df[
    (df["Grouping_Status"] == "REVIEW_REQUIRED")
    | (df["Thickness_Comparison_Status"] != "MATCH")
    | (~df["Depth_Interval_Valid"])
].copy()

# Save revised files
df.to_csv(MODEL / "Alwar_Hydrostratigraphic_Intervals.csv", index=False)
group_summary.to_csv(
    MODEL / "Alwar_Hydrostratigraphic_Group_Summary.csv", index=False
)
borehole_summary.to_csv(
    MODEL / "Alwar_Borehole_Hydrostratigraphic_Summary.csv", index=False
)
review.to_csv(
    MODEL / "Alwar_Hydrostratigraphic_Review.csv", index=False
)

print("Revised grouping completed.")
print("Intervals:", len(df))
print("Boreholes:", df["Borehole_ID"].nunique())
print("Unassigned intervals:",
      (df["Hydrostratigraphic_Group_ID"] == "UNASSIGNED").sum())
print("Review records:", len(review))

display(group_summary)
display(
    review[
        [
            "Borehole_ID", "Source_Row", "Source_Lithology",
            "Lithology_Class", "Hydrostratigraphic_Group_ID",
            "Thickness_Comparison_Status"
        ]
    ]
)

**[orig #280]** Inspect the elevation database and depth coverage per borehole  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
from pathlib import Path

MODEL = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input"
)

ELEV_FILE = (
    MODEL / "Step6_Elevation_Validation" /
    "Step6F_Borehole_Lithology_Elevation_Database.csv"
)

HS_FILE = MODEL / "Alwar_Hydrostratigraphic_Intervals.csv"

elev = pd.read_csv(ELEV_FILE)
hs = pd.read_csv(HS_FILE)

print("Elevation database columns:")
print(elev.columns.tolist())

print("\nHydrostratigraphic database columns:")
print(hs.columns.tolist())

# Inspect one record per borehole from elevation database
print("\nElevation database sample:")
display(elev.head(3))

# Summarise hydrostratigraphic depth coverage
depth_summary = (
    hs.groupby(["Borehole_ID", "Dataset"])
    .agg(
        Interval_Count=("Source_Row", "count"),
        Minimum_Depth_m=("Source_From_m", "min"),
        Maximum_Depth_m=("Source_To_m", "max"),
        HSG_Count=("Hydrostratigraphic_Group_ID", "nunique")
    )
    .reset_index()
)

# Identify coordinate/elevation fields available
candidate_fields = [
    c for c in elev.columns
    if any(term in c.lower() for term in [
        "latitude", "longitude", "elevation",
        "confidence", "source", "status"
    ])
]

print("\nCandidate coordinate/elevation fields:")
print(candidate_fields)

print("\nDepth coverage summary:")
display(depth_summary)

print("\nElevation records per borehole:")
display(
    elev.drop_duplicates("Borehole_ID")[
        ["Borehole_ID"] + candidate_fields
    ]
)

**[orig #282]** Inspect coordinate values for NHP and ATAL boreholes  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready"
)

MODEL = BASE / "MODFLOW_Model_Input"

STEP6 = MODEL / "Step6_Elevation_Validation"

COORD_FILE = STEP6 / "Step6B_Borehole_Coordinates_For_DEM.csv"
ELEV_FILE = STEP6 / "Step6F_Borehole_Lithology_Elevation_Database.csv"
HS_FILE = MODEL / "Alwar_Hydrostratigraphic_Intervals.csv"

for file in [COORD_FILE, ELEV_FILE, HS_FILE]:
    print("\nFile:", file)
    print("Exists:", file.exists())

    if not file.exists():
        continue

    data = pd.read_csv(file)
    print("Rows:", len(data))
    print("Columns:", data.columns.tolist())
    display(data.head(2))

# Inspect coordinate values for NHP and ATAL
coords = pd.read_csv(COORD_FILE)

print("\nCoordinate file borehole count:",
      coords["Borehole_ID"].nunique()
      if "Borehole_ID" in coords.columns else "Borehole_ID not found")

## 2. Correlation candidates (Steps 7C–7G)

**[orig #283]** Step 7C — borehole correlation register and candidate pairs  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from itertools import combinations

# =========================================================
# 1. PATHS
# =========================================================

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready"
)

MODEL = BASE / "MODFLOW_Model_Input"
STEP6 = MODEL / "Step6_Elevation_Validation"

COORD_FILE = STEP6 / "Step6B_Borehole_Coordinates_For_DEM.csv"
ELEV_FILE = STEP6 / "Step6F_Borehole_Lithology_Elevation_Database.csv"
HS_FILE = MODEL / "Alwar_Hydrostratigraphic_Intervals.csv"

OUT_REGISTER = MODEL / "Step7C_Borehole_Correlation_Register.csv"
OUT_MATRIX = MODEL / "Step7C_Borehole_Distance_Matrix.csv"
OUT_PAIRS = MODEL / "Step7C_Correlation_Candidate_Pairs.csv"

# =========================================================
# 2. LOAD DATA
# =========================================================

coords = pd.read_csv(COORD_FILE)
elev = pd.read_csv(ELEV_FILE)
hs = pd.read_csv(HS_FILE)

print("Coordinate records:", len(coords))
print("Elevation interval records:", len(elev))
print("Hydrostratigraphic intervals:", len(hs))

# Required fields
for col in ["Borehole_ID", "Latitude", "Longitude"]:
    if col not in coords.columns:
        raise ValueError(f"Missing coordinate field: {col}")

for col in [
    "Borehole_ID",
    "Ground_Elevation_Candidate_m",
    "Elevation_Source_Selected",
    "Elevation_Confidence_Status"
]:
    if col not in elev.columns:
        raise ValueError(f"Missing elevation field: {col}")

# Check unique coordinate IDs
if coords["Borehole_ID"].duplicated().any():
    raise ValueError("Duplicate borehole IDs in coordinate file.")

# Check coordinate ranges
coords["Latitude"] = pd.to_numeric(coords["Latitude"], errors="coerce")
coords["Longitude"] = pd.to_numeric(coords["Longitude"], errors="coerce")

if coords[["Latitude", "Longitude"]].isna().any().any():
    raise ValueError("Missing or non-numeric coordinates detected.")

if not coords["Latitude"].between(-90, 90).all():
    raise ValueError("Latitude outside valid geographic range.")

if not coords["Longitude"].between(-180, 180).all():
    raise ValueError("Longitude outside valid geographic range.")

# =========================================================
# 3. CREATE ONE ELEVATION RECORD PER BOREHOLE
# =========================================================

elevation_fields = [
    "Ground_RL_m",
    "SRTM30m_Elevation_m",
    "Elevation_Source_Selected",
    "Elevation_Confidence_Status",
    "Ground_Elevation_Candidate_m",
    "Suitable_For_Final_Model_Elevation"
]

available_elevation_fields = [
    c for c in elevation_fields if c in elev.columns
]

# Verify values are constant within each borehole
for col in available_elevation_fields:
    n_unique = elev.groupby("Borehole_ID")[col].nunique(dropna=True)
    inconsistent = n_unique[n_unique > 1]

    if len(inconsistent):
        raise ValueError(
            f"Inconsistent {col} values in boreholes: "
            f"{inconsistent.index.tolist()}"
        )

elev_bh = (
    elev[["Borehole_ID"] + available_elevation_fields]
    .drop_duplicates("Borehole_ID")
)

# =========================================================
# 4. HYDROSTRATIGRAPHIC SUMMARY PER BOREHOLE
# =========================================================

hs["Source_From_m"] = pd.to_numeric(
    hs["Source_From_m"], errors="coerce"
)

hs["Source_To_m"] = pd.to_numeric(
    hs["Source_To_m"], errors="coerce"
)

hs_sorted = hs.sort_values(
    ["Borehole_ID", "Source_From_m", "Source_To_m"]
).copy()

def make_sequence(group):
    result = []
    for row in group.itertuples():
        result.append(
            f"{row.Hydrostratigraphic_Group_ID}: "
            f"{row.Source_From_m:g}-{row.Source_To_m:g} m"
        )
    return " | ".join(result)

hs_summary = (
    hs_sorted.groupby("Borehole_ID")
    .agg(
        Dataset=("Dataset", "first"),
        Interval_Count=("Source_Row", "count"),
        Minimum_Depth_m=("Source_From_m", "min"),
        Maximum_Depth_m=("Source_To_m", "max"),
        HSG_Count=("Hydrostratigraphic_Group_ID", "nunique")
    )
    .reset_index()
)

sequence = (
    hs_sorted.groupby("Borehole_ID")
    .apply(make_sequence, include_groups=False)
    .rename("Hydrostratigraphic_Sequence")
    .reset_index()
)

hs_summary = hs_summary.merge(
    sequence, on="Borehole_ID", how="left"
)

hsg_counts = pd.crosstab(
    hs["Borehole_ID"],
    hs["Hydrostratigraphic_Group_ID"]
).reset_index()

hs_summary = hs_summary.merge(
    hsg_counts, on="Borehole_ID", how="left"
)

# =========================================================
# 5. CREATE MASTER BOREHOLE REGISTER
# =========================================================

register = coords.merge(
    elev_bh,
    on="Borehole_ID",
    how="left",
    validate="one_to_one"
)

register = register.merge(
    hs_summary,
    on="Borehole_ID",
    how="left",
    validate="one_to_one"
)

# Coordinate provenance warning
register["Coordinate_Validation_Status"] = "PENDING_INDEPENDENT_VERIFICATION"

register["Elevation_Use_Status"] = (
    "PROVISIONAL_NOT_FOR_FINAL_MODEL"
)

register["Correlation_Use_Status"] = (
    "SPATIAL_SCREENING_ONLY"
)

if register["Ground_Elevation_Candidate_m"].isna().any():
    print("WARNING: Some boreholes have missing elevation candidates.")

if register["Interval_Count"].isna().any():
    raise ValueError("Some coordinate boreholes lack lithology summaries.")

# Save register
register.to_csv(OUT_REGISTER, index=False)

# =========================================================
# 6. CALCULATE PAIRWISE DISTANCES
# =========================================================

# Haversine distance, using latitude/longitude in degrees
EARTH_RADIUS_KM = 6371.0088

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = np.radians(
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * EARTH_RADIUS_KM * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )

ids = register["Borehole_ID"].tolist()
n = len(register)

distance_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        distance_matrix[i, j] = haversine_km(
            register.iloc[i]["Latitude"],
            register.iloc[i]["Longitude"],
            register.iloc[j]["Latitude"],
            register.iloc[j]["Longitude"]
        )

distance_df = pd.DataFrame(
    distance_matrix,
    index=ids,
    columns=ids
)

distance_df.index.name = "Borehole_ID"
distance_df.to_csv(OUT_MATRIX)

# =========================================================
# 7. CREATE ALL BOREHOLE PAIRS
# =========================================================

pair_records = []

for i, j in combinations(range(n), 2):

    a = register.iloc[i]
    b = register.iloc[j]

    distance = distance_matrix[i, j]

    if distance <= 5:
        proximity = "VERY_NEAR_0_TO_5_KM"
    elif distance <= 10:
        proximity = "NEAR_5_TO_10_KM"
    elif distance <= 20:
        proximity = "INTERMEDIATE_10_TO_20_KM"
    else:
        proximity = "DISTANT_OVER_20_KM"

    pair_records.append({
        "Borehole_A": a["Borehole_ID"],
        "Borehole_B": b["Borehole_ID"],
        "Dataset_A": a["Dataset"],
        "Dataset_B": b["Dataset"],
        "Distance_km": round(distance, 3),
        "Proximity_Class": proximity,
        "Elevation_A_m": a["Ground_Elevation_Candidate_m"],
        "Elevation_B_m": b["Ground_Elevation_Candidate_m"],
        "Elevation_Source_A": a["Elevation_Source_Selected"],
        "Elevation_Source_B": b["Elevation_Source_Selected"],
        "Elevation_Confidence_A": a["Elevation_Confidence_Status"],
        "Elevation_Confidence_B": b["Elevation_Confidence_Status"],
        "Maximum_Depth_A_m": a["Maximum_Depth_m"],
        "Maximum_Depth_B_m": b["Maximum_Depth_m"],
        "HSG_Count_A": a["HSG_Count"],
        "HSG_Count_B": b["HSG_Count"],
        "Correlation_Status": "NOT_YET_GEOLOGICALLY_VALIDATED"
    })

pairs = pd.DataFrame(pair_records)

pairs = pairs.sort_values("Distance_km").reset_index(drop=True)

# This is a spatial screening subset, not a geological correlation decision.
pairs["Within_20km_Screening"] = np.where(
    pairs["Distance_km"] <= 20, "YES", "NO"
)

pairs.to_csv(OUT_PAIRS, index=False)

# =========================================================
# 8. NEAREST NEIGHBOUR SUMMARY
# =========================================================

nearest = []

for i in range(n):
    row = distance_matrix[i].copy()
    row[i] = np.inf

    j = np.argmin(row)

    nearest.append({
        "Borehole_ID": ids[i],
        "Nearest_Borehole_ID": ids[j],
        "Nearest_Distance_km": round(row[j], 3)
    })

nearest_df = pd.DataFrame(nearest)

register = register.merge(
    nearest_df, on="Borehole_ID", how="left"
)

register.to_csv(OUT_REGISTER, index=False)

# =========================================================
# 9. REPORT
# =========================================================

print("\n========== STEP 7C RESULTS ==========")

print("Boreholes in register:", len(register))
print("Pairwise distance records:", len(pairs))
print("Pairs within 5 km:", (pairs["Distance_km"] <= 5).sum())
print("Pairs within 10 km:", (pairs["Distance_km"] <= 10).sum())
print("Pairs within 20 km:", (pairs["Distance_km"] <= 20).sum())

print("\nCoordinate sources:")
display(
    register.groupby("Coordinate_Source")
    .size()
    .reset_index(name="Borehole_Count")
)

print("\nElevation confidence:")
display(
    register.groupby("Elevation_Confidence_Status")
    .size()
    .reset_index(name="Borehole_Count")
)

print("\nNearest borehole for each borehole:")
display(
    register[
        [
            "Borehole_ID",
            "Nearest_Borehole_ID",
            "Nearest_Distance_km",
            "Ground_Elevation_Candidate_m",
            "Elevation_Confidence_Status"
        ]
    ].sort_values("Nearest_Distance_km")
)

print("\nTen closest borehole pairs:")
display(pairs.head(10))

print("\nSaved outputs:")
print(OUT_REGISTER)
print(OUT_MATRIX)
print(OUT_PAIRS)

**[orig #284]** Step 7D — plot the seven closest borehole pairs  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Patch
import re

# =========================================================
# 1. PATHS
# =========================================================

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready"
)

MODEL = BASE / "MODFLOW_Model_Input"

HS_FILE = MODEL / "Alwar_Hydrostratigraphic_Intervals.csv"
PAIR_FILE = MODEL / "Step7C_Correlation_Candidate_Pairs.csv"

OUT_DIR = MODEL / "Step7D_Preliminary_Correlation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PDF_FILE = OUT_DIR / "Step7D_Seven_Closest_Borehole_Pairs.pdf"
INTERVAL_FILE = OUT_DIR / "Step7D_Selected_Pair_Intervals.csv"

# =========================================================
# 2. LOAD AND VALIDATE
# =========================================================

hs = pd.read_csv(HS_FILE)
pairs = pd.read_csv(PAIR_FILE)

required_hs = [
    "Borehole_ID",
    "Source_From_m",
    "Source_To_m",
    "Source_Lithology",
    "Lithology_Class",
    "Hydrostratigraphic_Group_ID"
]

for col in required_hs:
    if col not in hs.columns:
        raise ValueError(f"Missing field: {col}")

selected = (
    pairs[pairs["Distance_km"] <= 5]
    .sort_values("Distance_km")
    .copy()
)

if len(selected) == 0:
    raise ValueError("No borehole pairs found within 5 km.")

print("Selected pairs:", len(selected))
display(
    selected[
        ["Borehole_A", "Borehole_B", "Distance_km"]
    ]
)

# =========================================================
# 3. GROUP COLOURS
# =========================================================

GROUP_COLORS = {
    "HSG-1": "#E6B85C",
    "HSG-2": "#8FBBD9",
    "HSG-3": "#91B77A",
    "HSG-4": "#A69BBE",
    "HSG-5": "#C7B49A",
    "UNASSIGNED": "#D0D0D0"
}

GROUP_NAMES = {
    "HSG-1": "Granular deposits",
    "HSG-2": "Fine-grained deposits",
    "HSG-3": "Weathered rock",
    "HSG-4": "Consolidated bedrock",
    "HSG-5": "Unspecified soil"
}

# =========================================================
# 4. PLOT FUNCTION
# =========================================================

def plot_borehole(ax, borehole_id, x_position):

    data = hs[hs["Borehole_ID"] == borehole_id].copy()

    data["Source_From_m"] = pd.to_numeric(
        data["Source_From_m"], errors="coerce"
    )
    data["Source_To_m"] = pd.to_numeric(
        data["Source_To_m"], errors="coerce"
    )

    data = data.sort_values("Source_From_m")

    if data.empty:
        raise ValueError(f"No intervals found for {borehole_id}")

    for _, row in data.iterrows():

        top = row["Source_From_m"]
        bottom = row["Source_To_m"]
        group = row["Hydrostratigraphic_Group_ID"]

        if pd.isna(top) or pd.isna(bottom) or bottom <= top:
            continue

        color = GROUP_COLORS.get(group, "#D0D0D0")

        ax.bar(
            x_position,
            bottom - top,
            bottom=top,
            width=0.65,
            color=color,
            edgecolor="black",
            linewidth=0.7
        )

        # Label only where interval is tall enough
        if bottom - top >= 5:
            ax.text(
                x_position,
                (top + bottom) / 2,
                str(group),
                ha="center",
                va="center",
                fontsize=8,
                fontweight="bold"
            )

    maximum_depth = data["Source_To_m"].max()

    ax.set_xlim(x_position - 0.65, x_position + 0.65)
    ax.set_ylim(maximum_depth + 5, 0)
    ax.set_xticks([x_position])
    ax.set_xticklabels([borehole_id], rotation=15, ha="right", fontsize=8)
    ax.set_ylabel("Depth below ground level (m bgl)")
    ax.grid(axis="y", linestyle="--", alpha=0.35)

    return maximum_depth

# =========================================================
# 5. CREATE PDF AND PNG PANELS
# =========================================================

interval_records = []

legend_handles = [
    Patch(
        facecolor=GROUP_COLORS[group],
        edgecolor="black",
        label=f"{group}: {GROUP_NAMES[group]}"
    )
    for group in GROUP_COLORS
    if group in GROUP_NAMES
]

with PdfPages(PDF_FILE) as pdf:

    for pair_number, row in enumerate(
        selected.itertuples(index=False), start=1
    ):

        borehole_a = row.Borehole_A
        borehole_b = row.Borehole_B
        distance = row.Distance_km

        fig, axes = plt.subplots(
            1, 2,
            figsize=(12, 8),
            sharey=False
        )

        depth_a = plot_borehole(axes[0], borehole_a, 0)
        depth_b = plot_borehole(axes[1], borehole_b, 1)

        axes[0].set_title(borehole_a, fontsize=10)
        axes[1].set_title(borehole_b, fontsize=10)

        fig.suptitle(
            f"Preliminary lithological comparison — Pair {pair_number}\n"
            f"Distance: {distance:.3f} km | "
            "Depth referenced to individual ground surface",
            fontsize=13
        )

        fig.legend(
            handles=legend_handles,
            loc="lower center",
            ncol=3,
            fontsize=8
        )

        fig.text(
            0.5, 0.035,
            "Exploratory comparison only. No geological continuity "
            "or aquifer connection is implied.",
            ha="center",
            fontsize=9
        )

        fig.tight_layout(rect=[0, 0.12, 1, 0.90])

        png_file = OUT_DIR / (
            f"Pair_{pair_number:02d}_"
            f"{re.sub(r'[^A-Za-z0-9]+', '_', borehole_a)}_"
            f"{re.sub(r'[^A-Za-z0-9]+', '_', borehole_b)}.png"
        )

        fig.savefig(png_file, dpi=200, bbox_inches="tight")
        pdf.savefig(fig)
        plt.close(fig)

        # Export interval-level records for both boreholes
        for bh in [borehole_a, borehole_b]:
            temp = hs[hs["Borehole_ID"] == bh].copy()
            temp["Pair_Number"] = pair_number
            temp["Pair_Distance_km"] = distance
            temp["Pair_Borehole_A"] = borehole_a
            temp["Pair_Borehole_B"] = borehole_b
            interval_records.append(temp)

# =========================================================
# 6. SAVE INTERVAL COMPARISON DATA
# =========================================================

pair_intervals = pd.concat(
    interval_records,
    ignore_index=True
)

pair_intervals.to_csv(INTERVAL_FILE, index=False)

print("\n========== STEP 7D RESULTS ==========")
print("Pairs plotted:", len(selected))
print("Interval records exported:", len(pair_intervals))
print("PDF:", PDF_FILE)
print("Interval CSV:", INTERVAL_FILE)
print("PNG panels:", OUT_DIR)

**[orig #285]** Step 7E — preliminary correlation review (screening only)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ============================================================
# STEP 7E — PRELIMINARY HYDROSTRATIGRAPHIC CORRELATION REVIEW
# Source-preserving screening only. No automatic layer correlation.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

BASE = (
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input"
)

PAIR_FILE = os.path.join(
    BASE, "Step7C_Correlation_Candidate_Pairs.csv"
)

HSG_FILE = os.path.join(
    BASE, "Alwar_Hydrostratigraphic_Intervals.csv"
)

ELEV_FILE = os.path.join(
    BASE, "Step6_Elevation_Validation",
    "Step6F_Borehole_Lithology_Elevation_Database.csv"
)

OUT = os.path.join(BASE, "Step7E_Correlation_Review")
os.makedirs(OUT, exist_ok=True)

# ------------------------------------------------------------
# 1. LOAD INPUTS
# ------------------------------------------------------------

pairs = pd.read_csv(PAIR_FILE)
hsg = pd.read_csv(HSG_FILE)
elev = pd.read_csv(ELEV_FILE)

print("Pairs:", len(pairs))
print("HSG intervals:", len(hsg))
print("Elevation records:", len(elev))

# ------------------------------------------------------------
# 2. CHECK REQUIRED FIELDS
# ------------------------------------------------------------

required_pairs = ["Borehole_A", "Borehole_B", "Distance_km"]
required_hsg = [
    "Borehole_ID",
    "Hydrostratigraphic_Group_ID"
]

for c in required_pairs:
    if c not in pairs.columns:
        raise ValueError(f"Missing pair column: {c}")

for c in required_hsg:
    if c not in hsg.columns:
        raise ValueError(f"Missing HSG column: {c}")

# Identify depth columns without assuming their exact names.
def find_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(
        "Could not find any of these columns: " + str(candidates)
    )

top_col = find_column(
    hsg, ["Source_From_m", "Depth_From_m_bgl", "From_m"]
)
bottom_col = find_column(
    hsg, ["Source_To_m", "Depth_To_m_bgl", "To_m"]
)

print("Depth top field:", top_col)
print("Depth bottom field:", bottom_col)

hsg[top_col] = pd.to_numeric(hsg[top_col], errors="coerce")
hsg[bottom_col] = pd.to_numeric(hsg[bottom_col], errors="coerce")

# ------------------------------------------------------------
# 3. PREPARE ELEVATION SUMMARY
# ------------------------------------------------------------

# Elevation file may contain multiple interval rows per borehole.
# Keep one consistent elevation record per borehole.
elev_cols = [
    "Borehole_ID",
    "Ground_Elevation_Candidate_m",
    "Elevation_Source_Selected",
    "Elevation_Confidence_Status"
]

missing = [c for c in elev_cols if c not in elev.columns]
if missing:
    raise ValueError(f"Missing elevation fields: {missing}")

elev_summary = (
    elev[elev_cols]
    .drop_duplicates()
    .drop_duplicates(subset=["Borehole_ID"])
)

# Candidate interval elevations from ground elevation minus depth.
hsg = hsg.merge(
    elev_summary,
    on="Borehole_ID",
    how="left",
    validate="many_to_one"
)

hsg["Candidate_Z_Top_m"] = (
    hsg["Ground_Elevation_Candidate_m"] - hsg[top_col]
)

hsg["Candidate_Z_Bottom_m"] = (
    hsg["Ground_Elevation_Candidate_m"] - hsg[bottom_col]
)

# ------------------------------------------------------------
# 4. CREATE INTERVAL-PAIR SCREENING TABLE
# ------------------------------------------------------------

records = []

for _, p in pairs.iterrows():

    a = p["Borehole_A"]
    b = p["Borehole_B"]
    distance = p["Distance_km"]

    A = hsg[hsg["Borehole_ID"] == a]
    B = hsg[hsg["Borehole_ID"] == b]

    for _, ia in A.iterrows():
        for _, ib in B.iterrows():

            group_a = ia["Hydrostratigraphic_Group_ID"]
            group_b = ib["Hydrostratigraphic_Group_ID"]

            # Depth overlap is a comparison in m below ground level.
            depth_overlap = max(
                0,
                min(ia[bottom_col], ib[bottom_col]) -
                max(ia[top_col], ib[top_col])
            )

            # Candidate elevation overlap is exploratory only.
            z_values = [
                ia["Candidate_Z_Top_m"],
                ia["Candidate_Z_Bottom_m"],
                ib["Candidate_Z_Top_m"],
                ib["Candidate_Z_Bottom_m"]
            ]

            if all(pd.notna(x) for x in z_values):
                a_z_low = min(
                    ia["Candidate_Z_Top_m"],
                    ia["Candidate_Z_Bottom_m"]
                )
                a_z_high = max(
                    ia["Candidate_Z_Top_m"],
                    ia["Candidate_Z_Bottom_m"]
                )
                b_z_low = min(
                    ib["Candidate_Z_Top_m"],
                    ib["Candidate_Z_Bottom_m"]
                )
                b_z_high = max(
                    ib["Candidate_Z_Top_m"],
                    ib["Candidate_Z_Bottom_m"]
                )

                elevation_overlap = max(
                    0,
                    min(a_z_high, b_z_high) -
                    max(a_z_low, b_z_low)
                )
            else:
                elevation_overlap = np.nan

            same_group = (
                pd.notna(group_a) and
                pd.notna(group_b) and
                group_a == group_b
            )

            # Retain all comparisons, but flag only same-group
            # comparisons as candidates for closer inspection.
            if same_group:
                screening_status = (
                    "SAME_HSG_GROUP_REVIEW"
                    if depth_overlap > 0
                    else "SAME_HSG_GROUP_NO_DEPTH_OVERLAP"
                )
            else:
                screening_status = "DIFFERENT_HSG_GROUPS"

            records.append({
                "Borehole_A": a,
                "Borehole_B": b,
                "Distance_km": distance,

                "HSG_A": group_a,
                "HSG_B": group_b,

                "A_From_m_bgl": ia[top_col],
                "A_To_m_bgl": ia[bottom_col],
                "B_From_m_bgl": ib[top_col],
                "B_To_m_bgl": ib[bottom_col],

                "Depth_Overlap_m": depth_overlap,
                "Same_HSG_Group": same_group,

                "A_Candidate_Z_Top_m": ia["Candidate_Z_Top_m"],
                "A_Candidate_Z_Bottom_m": ia["Candidate_Z_Bottom_m"],
                "B_Candidate_Z_Top_m": ib["Candidate_Z_Top_m"],
                "B_Candidate_Z_Bottom_m": ib["Candidate_Z_Bottom_m"],

                "Candidate_Elevation_Overlap_m": elevation_overlap,

                "Elevation_Source_A": ia["Elevation_Source_Selected"],
                "Elevation_Source_B": ib["Elevation_Source_Selected"],
                "Elevation_Confidence_A": ia["Elevation_Confidence_Status"],
                "Elevation_Confidence_B": ib["Elevation_Confidence_Status"],

                "Screening_Status": screening_status,

                "Scientific_Review": "PENDING",
                "Correlation_Accepted": "NO",
                "Reviewer_Comments": ""
            })

review = pd.DataFrame(records)

# ------------------------------------------------------------
# 5. CREATE PAIR-LEVEL SUMMARY
# ------------------------------------------------------------

summary = (
    review.groupby(
        ["Borehole_A", "Borehole_B", "Distance_km"],
        dropna=False
    )
    .agg(
        Interval_Comparisons=("Screening_Status", "size"),
        Same_HSG_Comparisons=("Same_HSG_Group", "sum"),
        Positive_Depth_Overlap_Count=(
            "Depth_Overlap_m",
            lambda x: int((x > 0).sum())
        ),
        Positive_Elevation_Overlap_Count=(
            "Candidate_Elevation_Overlap_m",
            lambda x: int((x > 0).sum())
        )
    )
    .reset_index()
)

summary["Interpretation_Status"] = (
    "EXPLORATORY_SCREENING_ONLY"
)

# ------------------------------------------------------------
# 6. SAVE OUTPUTS
# ------------------------------------------------------------

review_file = os.path.join(
    OUT, "Step7E_Interval_Pair_Review.csv"
)

summary_file = os.path.join(
    OUT, "Step7E_Borehole_Pair_Summary.csv"
)

excel_file = os.path.join(
    OUT, "Step7E_Correlation_Review_Workbook.xlsx"
)

review.to_csv(review_file, index=False)
summary.to_csv(summary_file, index=False)

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    summary.to_excel(
        writer, sheet_name="Pair_Summary", index=False
    )
    review.to_excel(
        writer, sheet_name="Interval_Comparisons", index=False
    )

print("\n========== STEP 7E COMPLETED ==========")
print("Borehole pairs:", len(summary))
print("Interval comparisons:", len(review))
print(
    "Same HSG comparisons:",
    int(review["Same_HSG_Group"].sum())
)
print("Review workbook:", excel_file)
print("Interval CSV:", review_file)
print("Pair summary:", summary_file)


**[orig #286]** Step 7E-1 — focused review of the seven pairs within 5 km  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import os
import pandas as pd
import numpy as np

BASE = (
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input"
)

INPUT = os.path.join(
    BASE, "Step7E_Correlation_Review",
    "Step7E_Interval_Pair_Review.csv"
)

OUT = os.path.join(
    BASE, "Step7E_Correlation_Review",
    "Step7E_Seven_Pair_Focused_Review.xlsx"
)

df = pd.read_csv(INPUT)

# Select the seven pairs within 5 km.
focused = df[df["Distance_km"] <= 5.0].copy()

# Pair-level summary.
summary = (
    focused.groupby(
        ["Borehole_A", "Borehole_B", "Distance_km"]
    )
    .agg(
        Interval_Comparisons=("Screening_Status", "size"),
        Same_HSG_Comparisons=("Same_HSG_Group", "sum"),
        Positive_Depth_Overlap=(
            "Depth_Overlap_m",
            lambda x: int((x > 0).sum())
        ),
        Positive_Elevation_Overlap=(
            "Candidate_Elevation_Overlap_m",
            lambda x: int((x > 0).sum())
        )
    )
    .reset_index()
)

# Retain comparisons with matching HSG groups.
same_group = focused[
    focused["Same_HSG_Group"] == True
].copy()

# Among these, identify candidate elevation overlaps.
elevation_candidates = same_group[
    same_group["Candidate_Elevation_Overlap_m"] > 0
].copy()

# Sort for review; this is a screening order, not a ranking
# of geological validity.
elevation_candidates = elevation_candidates.sort_values(
    ["Borehole_A", "Borehole_B",
     "Candidate_Elevation_Overlap_m"],
    ascending=[True, True, False]
)

with pd.ExcelWriter(OUT, engine="openpyxl") as writer:
    summary.to_excel(
        writer, sheet_name="Seven_Pair_Summary", index=False
    )
    focused.to_excel(
        writer, sheet_name="All_Interval_Comparisons", index=False
    )
    same_group.to_excel(
        writer, sheet_name="Same_HSG_Comparisons", index=False
    )
    elevation_candidates.to_excel(
        writer, sheet_name="Elevation_Overlap_Candidates",
        index=False
    )

print("========== STEP 7E-1 COMPLETED ==========")
print("Selected pairs:", summary.shape[0])
print("Interval comparisons:", len(focused))
print("Same HSG comparisons:", len(same_group))
print("Positive candidate elevation overlaps:",
      len(elevation_candidates))
print("\nPAIR SUMMARY")
display(summary)

print("\nSaved:", OUT)


**[orig #287]** Step 7F — source-based validation of candidate correlations  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ============================================================
# STEP 7F — SOURCE-BASED CANDIDATE CORRELATION VALIDATION
# No automatic geological correlation or MODFLOW layer assignment
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

BASE = (
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input"
)

FOCUSED_FILE = os.path.join(
    BASE, "Step7E_Correlation_Review",
    "Step7E_Seven_Pair_Focused_Review.xlsx"
)

HSG_FILE = os.path.join(
    BASE, "Alwar_Hydrostratigraphic_Intervals.csv"
)

OUT_DIR = os.path.join(BASE, "Step7F_Correlation_Validation")
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

candidates = pd.read_excel(
    FOCUSED_FILE,
    sheet_name="Elevation_Overlap_Candidates"
)

hsg = pd.read_csv(HSG_FILE)

print("Candidate records:", len(candidates))
print("Hydrostratigraphic intervals:", len(hsg))

# ------------------------------------------------------------
# 2. CHECK REQUIRED FIELDS
# ------------------------------------------------------------

required_candidate = [
    "Borehole_A", "Borehole_B", "Distance_km",
    "HSG_A", "HSG_B",
    "A_From_m_bgl", "A_To_m_bgl",
    "B_From_m_bgl", "B_To_m_bgl",
    "Depth_Overlap_m",
    "Candidate_Elevation_Overlap_m",
    "Elevation_Confidence_A",
    "Elevation_Confidence_B"
]

missing = [c for c in required_candidate if c not in candidates.columns]

if missing:
    raise ValueError(
        f"Missing candidate columns: {missing}"
    )

required_hsg = [
    "Borehole_ID",
    "Source_From_m",
    "Source_To_m",
    "Hydrostratigraphic_Group_ID"
]

missing = [c for c in required_hsg if c not in hsg.columns]

if missing:
    raise ValueError(
        f"Missing HSG columns: {missing}"
    )

# ------------------------------------------------------------
# 3. IDENTIFY SOURCE DESCRIPTION AND CLASSIFICATION FIELDS
# ------------------------------------------------------------

def find_optional_column(df, possible_names):
    for col in possible_names:
        if col in df.columns:
            return col
    return None

lithology_col = find_optional_column(
    hsg,
    [
        "Source_Lithology",
        "Original_Lithology",
        "Lithology_Description",
        "Lithology_Raw",
        "Lithology_Description_Original",
        "Lithology"
    ]
)

class_col = find_optional_column(
    hsg,
    [
        "Final_Lithology_Class",
        "Lithology_Class",
        "Lithology_Class_Final",
        "Simplified_Lithology_Class"
    ]
)

source_row_col = find_optional_column(
    hsg,
    ["Source_Row", "Source_Row_Number", "Original_Source_Row"]
)

print("Lithology description field:", lithology_col)
print("Lithology class field:", class_col)
print("Source row field:", source_row_col)

if lithology_col is None:
    print(
        "\nWARNING: Original lithology description field was not "
        "identified. Inspect HSG columns before interpreting candidates."
    )
    print(hsg.columns.tolist())

# ------------------------------------------------------------
# 4. PREPARE SOURCE INTERVAL DATABASE
# ------------------------------------------------------------

hsg["Source_From_m"] = pd.to_numeric(
    hsg["Source_From_m"], errors="coerce"
)

hsg["Source_To_m"] = pd.to_numeric(
    hsg["Source_To_m"], errors="coerce"
)

# Keep source descriptions and source identifiers unchanged.
keep_cols = [
    "Borehole_ID",
    "Source_From_m",
    "Source_To_m",
    "Hydrostratigraphic_Group_ID"
]

for c in [lithology_col, class_col, source_row_col]:
    if c and c not in keep_cols:
        keep_cols.append(c)

source_intervals = hsg[keep_cols].copy()

# ------------------------------------------------------------
# 5. ENRICH BOREHOLE A AND B INTERVALS
# ------------------------------------------------------------

def attach_source_intervals(df, borehole_col, from_col, to_col, prefix):

    left = df.copy()

    left["_top_key"] = pd.to_numeric(
        left[from_col], errors="coerce"
    ).round(5)

    left["_bottom_key"] = pd.to_numeric(
        left[to_col], errors="coerce"
    ).round(5)

    right = source_intervals.copy()

    right["_top_key"] = right["Source_From_m"].round(5)
    right["_bottom_key"] = right["Source_To_m"].round(5)

    right = right.rename(columns={
        "Borehole_ID": borehole_col,
        "Hydrostratigraphic_Group_ID": prefix + "_Source_HSG",
        "Source_From_m": prefix + "_Source_From_m",
        "Source_To_m": prefix + "_Source_To_m"
    })

    rename = {}

    if lithology_col:
        rename[lithology_col] = prefix + "_Original_Lithology"

    if class_col:
        rename[class_col] = prefix + "_Lithology_Class"

    if source_row_col:
        rename[source_row_col] = prefix + "_Source_Row"

    right = right.rename(columns=rename)

    # Avoid duplicate column names in the right-hand table.
    right_cols = [
        borehole_col, "_top_key", "_bottom_key",
        prefix + "_Source_HSG",
        prefix + "_Source_From_m",
        prefix + "_Source_To_m"
    ]

    for c in [
        prefix + "_Original_Lithology",
        prefix + "_Lithology_Class",
        prefix + "_Source_Row"
    ]:
        if c in right.columns:
            right_cols.append(c)

    right = right[right_cols].drop_duplicates()

    result = left.merge(
        right,
        left_on=[borehole_col, "_top_key", "_bottom_key"],
        right_on=[borehole_col, "_top_key", "_bottom_key"],
        how="left",
        validate="many_to_one"
    )

    result = result.drop(
        columns=["_top_key", "_bottom_key"],
        errors="ignore"
    )

    return result


# Attach source records for both boreholes.
review = attach_source_intervals(
    candidates,
    "Borehole_A",
    "A_From_m_bgl",
    "A_To_m_bgl",
    "A"
)

review = attach_source_intervals(
    review,
    "Borehole_B",
    "B_From_m_bgl",
    "B_To_m_bgl",
    "B"
)

# ------------------------------------------------------------
# 6. SOURCE MATCH AND GROUP CONSISTENCY CHECKS
# ------------------------------------------------------------

review["A_Source_Matched"] = review["A_Source_HSG"].notna()
review["B_Source_Matched"] = review["B_Source_HSG"].notna()

review["Source_HSG_Agrees"] = (
    review["HSG_A"].astype(str) ==
    review["A_Source_HSG"].astype(str)
) & review["A_Source_Matched"]

review["Source_HSG_B_Agrees"] = (
    review["HSG_B"].astype(str) ==
    review["B_Source_HSG"].astype(str)
) & review["B_Source_Matched"]

review["Both_Source_Intervals_Matched"] = (
    review["A_Source_Matched"] &
    review["B_Source_Matched"]
)

review["Both_HSG_Values_Verified"] = (
    review["Source_HSG_Agrees"] &
    review["Source_HSG_B_Agrees"]
)

# ------------------------------------------------------------
# 7. CREATE MANUAL GEOLOGICAL DECISION REGISTER
# ------------------------------------------------------------

review["Preliminary_Evidence_Status"] = np.where(
    review["Both_HSG_Values_Verified"],
    "SOURCE_INTERVALS_AND_HSG_MATCHED",
    "SOURCE_MATCH_OR_HSG_VERIFICATION_REQUIRED"
)

review["Geological_Correlation_Decision"] = "PENDING_REVIEW"

review["Aquifer_Aquitard_Interpretation"] = (
    "NOT_ASSIGNED"
)

review["MODFLOW_Layer_Assignment"] = (
    "NOT_ASSIGNED"
)

review["Reviewer_Comments"] = ""

review["Evidence_Required"] = (
    "Review original lithology, geological setting, "
    "water-bearing zones and elevation confidence"
)

# ------------------------------------------------------------
# 8. PAIR SUMMARY
# ------------------------------------------------------------

pair_summary = (
    review.groupby(
        ["Borehole_A", "Borehole_B", "Distance_km"],
        dropna=False
    )
    .agg(
        Candidate_Interval_Pairs=("Borehole_A", "size"),
        Both_Source_Intervals_Matched=(
            "Both_Source_Intervals_Matched", "sum"
        ),
        Both_HSG_Values_Verified=(
            "Both_HSG_Values_Verified", "sum"
        ),
        Mean_Elevation_Overlap_m=(
            "Candidate_Elevation_Overlap_m", "mean"
        ),
        Maximum_Elevation_Overlap_m=(
            "Candidate_Elevation_Overlap_m", "max"
        )
    )
    .reset_index()
)

pair_summary["Final_Correlation_Status"] = "PENDING_GEOLOGICAL_REVIEW"

# ------------------------------------------------------------
# 9. SOURCE MATCH EXCEPTIONS
# ------------------------------------------------------------

exceptions = review[
    ~review["Both_Source_Intervals_Matched"] |
    ~review["Both_HSG_Values_Verified"]
].copy()

# ------------------------------------------------------------
# 10. SAVE OUTPUTS
# ------------------------------------------------------------

csv_file = os.path.join(
    OUT_DIR, "Step7F_Candidate_Correlation_Validation.csv"
)

summary_file = os.path.join(
    OUT_DIR, "Step7F_Borehole_Pair_Validation_Summary.csv"
)

exception_file = os.path.join(
    OUT_DIR, "Step7F_Source_Match_Exceptions.csv"
)

excel_file = os.path.join(
    OUT_DIR, "Step7F_Correlation_Validation_Workbook.xlsx"
)

review.to_csv(csv_file, index=False)
pair_summary.to_csv(summary_file, index=False)
exceptions.to_csv(exception_file, index=False)

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    pair_summary.to_excel(
        writer, sheet_name="Pair_Summary", index=False
    )
    review.to_excel(
        writer, sheet_name="Candidate_Validation", index=False
    )
    exceptions.to_excel(
        writer, sheet_name="Source_Match_Exceptions", index=False
    )

# ------------------------------------------------------------
# 11. REPORT
# ------------------------------------------------------------

print("\n========== STEP 7F COMPLETED ==========")
print("Candidate interval pairs:", len(review))
print(
    "Both source intervals matched:",
    int(review["Both_Source_Intervals_Matched"].sum())
)
print(
    "Both HSG values verified:",
    int(review["Both_HSG_Values_Verified"].sum())
)
print("Source match exceptions:", len(exceptions))
print("Borehole pairs:", len(pair_summary))

print("\nPAIR SUMMARY")
display(pair_summary)

print("\nSOURCE MATCH EXCEPTIONS")
display(exceptions.head(20))

print("\nExcel workbook:", excel_file)
print("Candidate CSV:", csv_file)
print("Exception CSV:", exception_file)


**[orig #288]** Step 7F — depth boundary audit  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import pandas as pd
import numpy as np
import os

BASE = (
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input"
)

FILE = os.path.join(
    BASE, "Step7F_Correlation_Validation",
    "Step7F_Candidate_Correlation_Validation.csv"
)

df = pd.read_csv(FILE)

# Compare source boundaries with the candidate interval boundaries.
df["A_Top_Difference_m"] = (
    df["A_Source_From_m"] - df["A_From_m_bgl"]
).abs()

df["A_Bottom_Difference_m"] = (
    df["A_Source_To_m"] - df["A_To_m_bgl"]
).abs()

df["B_Top_Difference_m"] = (
    df["B_Source_From_m"] - df["B_From_m_bgl"]
).abs()

df["B_Bottom_Difference_m"] = (
    df["B_Source_To_m"] - df["B_To_m_bgl"]
).abs()

diff_cols = [
    "A_Top_Difference_m",
    "A_Bottom_Difference_m",
    "B_Top_Difference_m",
    "B_Bottom_Difference_m"
]

df["Depth_Boundaries_Exact"] = (
    df[diff_cols].fillna(np.inf).max(axis=1) <= 0.00001
)

print("========== STEP 7F DEPTH BOUNDARY AUDIT ==========")
print("Candidate records:", len(df))
print("Exact boundary matches:",
      int(df["Depth_Boundaries_Exact"].sum()))
print("Boundary mismatches:",
      int((~df["Depth_Boundaries_Exact"]).sum()))

display(df[
    [
        "Borehole_A", "Borehole_B",
        "A_Source_From_m", "A_Source_To_m",
        "B_Source_From_m", "B_Source_To_m",
        *diff_cols, "Depth_Boundaries_Exact"
    ]
])

audit_file = os.path.join(
    BASE, "Step7F_Correlation_Validation",
    "Step7F_Depth_Boundary_Audit.csv"
)

df.to_csv(audit_file, index=False)
print("Saved:", audit_file)


**[orig #289]** Step 7G — geological correlation assessment register  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ============================================================
# STEP 7G — GEOLOGICAL CORRELATION ASSESSMENT
# Source-preserving manual interpretation register
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

BASE = (
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input"
)

INPUT = os.path.join(
    BASE, "Step7F_Correlation_Validation",
    "Step7F_Candidate_Correlation_Validation.csv"
)

OUT_DIR = os.path.join(
    BASE, "Step7G_Geological_Assessment"
)
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT)

print("Input candidate records:", len(df))

# ------------------------------------------------------------
# 1. VERIFY REQUIRED SOURCE FIELDS
# ------------------------------------------------------------

required = [
    "Borehole_A", "Borehole_B", "Distance_km",
    "A_From_m_bgl", "A_To_m_bgl",
    "B_From_m_bgl", "B_To_m_bgl",
    "HSG_A", "HSG_B",
    "A_Original_Lithology",
    "B_Original_Lithology",
    "Candidate_Elevation_Overlap_m",
    "Elevation_Confidence_A",
    "Elevation_Confidence_B"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        "Required columns missing from Step 7F file: "
        + str(missing)
    )

# ------------------------------------------------------------
# 2. CREATE GEOLOGICAL REVIEW REGISTER
# ------------------------------------------------------------

review = df.copy()

# Add source lithology comparison fields.
review["Lithology_Description_A"] = (
    review["A_Original_Lithology"]
)

review["Lithology_Description_B"] = (
    review["B_Original_Lithology"]
)

review["HSG_Group_Match"] = (
    review["HSG_A"].astype(str) ==
    review["HSG_B"].astype(str)
)

review["Original_Description_Available"] = (
    review["Lithology_Description_A"].notna()
)

review["Original_Description_B_Available"] = (
    review["Lithology_Description_B"].notna()
)

review["Both_Descriptions_Available"] = (
    review["Original_Description_Available"] &
    review["Original_Description_B_Available"]
)

# ------------------------------------------------------------
# 3. CREATE EVIDENCE FLAGS
# ------------------------------------------------------------

review["Positive_Elevation_Overlap"] = (
    pd.to_numeric(
        review["Candidate_Elevation_Overlap_m"],
        errors="coerce"
    ) > 0
)

review["Same_HSG"] = review["HSG_Group_Match"]

review["Elevation_Confidence_Comparable"] = (
    review["Elevation_Confidence_A"].notna() &
    review["Elevation_Confidence_B"].notna()
)

# ------------------------------------------------------------
# 4. CREATE MANUAL DECISION FIELDS
# ------------------------------------------------------------

review["Lithology_Similarity_Assessment"] = "PENDING"

review["Stratigraphic_Position_Assessment"] = "PENDING"

review["Geological_Correlation_Decision"] = (
    "PENDING_MANUAL_REVIEW"
)

review["Correlation_Confidence"] = "NOT_ASSESSED"

review["Aquifer_Aquitard_Interpretation"] = "NOT_ASSIGNED"

review["MODFLOW_Layer_Assignment"] = "NOT_ASSIGNED"

review["Evidence_And_Reasoning"] = ""

review["Reviewer_Comments"] = ""

# ------------------------------------------------------------
# 5. CREATE REVIEW CATEGORY
# ------------------------------------------------------------

def review_category(row):

    if not row["Both_Descriptions_Available"]:
        return "SOURCE_DESCRIPTION_REQUIRED"

    if not row["HSG_Group_Match"]:
        return "DIFFERENT_HSG_GROUPS_REVIEW"

    if not row["Positive_Elevation_Overlap"]:
        return "NO_CANDIDATE_ELEVATION_OVERLAP"

    return "PRIORITY_GEOLOGICAL_REVIEW"

review["Review_Category"] = review.apply(
    review_category, axis=1
)

# ------------------------------------------------------------
# 6. PREPARE COMPACT GEOLOGICAL COMPARISON TABLE
# ------------------------------------------------------------

comparison_columns = [
    "Borehole_A",
    "Borehole_B",
    "Distance_km",

    "A_From_m_bgl",
    "A_To_m_bgl",
    "B_From_m_bgl",
    "B_To_m_bgl",

    "Lithology_Description_A",
    "Lithology_Description_B",

    "HSG_A",
    "HSG_B",
    "HSG_Group_Match",

    "Candidate_Elevation_Overlap_m",
    "Elevation_Confidence_A",
    "Elevation_Confidence_B",

    "Review_Category",
    "Geological_Correlation_Decision",
    "Correlation_Confidence",
    "Evidence_And_Reasoning",
    "Reviewer_Comments"
]

comparison = review[comparison_columns].copy()

# ------------------------------------------------------------
# 7. PAIR-LEVEL REVIEW SUMMARY
# ------------------------------------------------------------

pair_summary = (
    review.groupby(
        ["Borehole_A", "Borehole_B", "Distance_km"],
        dropna=False
    )
    .agg(
        Candidate_Intervals=("Borehole_A", "size"),
        Same_HSG_Count=("HSG_Group_Match", "sum"),
        Elevation_Overlap_Count=(
            "Positive_Elevation_Overlap", "sum"
        ),
        Original_Descriptions_Available=(
            "Both_Descriptions_Available", "sum"
        )
    )
    .reset_index()
)

pair_summary["Geological_Review_Status"] = (
    "PENDING_MANUAL_ASSESSMENT"
)

# ------------------------------------------------------------
# 8. SAVE FILES
# ------------------------------------------------------------

csv_file = os.path.join(
    OUT_DIR, "Step7G_Geological_Candidate_Register.csv"
)

comparison_file = os.path.join(
    OUT_DIR, "Step7G_Lithology_Comparison.csv"
)

summary_file = os.path.join(
    OUT_DIR, "Step7G_Pair_Review_Summary.csv"
)

excel_file = os.path.join(
    OUT_DIR, "Step7G_Geological_Assessment_Workbook.xlsx"
)

review.to_csv(csv_file, index=False)
comparison.to_csv(comparison_file, index=False)
pair_summary.to_csv(summary_file, index=False)

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:

    pair_summary.to_excel(
        writer,
        sheet_name="Pair_Summary",
        index=False
    )

    comparison.to_excel(
        writer,
        sheet_name="Lithology_Comparison",
        index=False
    )

    review.to_excel(
        writer,
        sheet_name="Full_Review_Register",
        index=False
    )

# ------------------------------------------------------------
# 9. REPORT
# ------------------------------------------------------------

print("\n========== STEP 7G COMPLETED ==========")

print("Candidate comparisons:", len(review))

print(
    "Both original descriptions available:",
    int(review["Both_Descriptions_Available"].sum())
)

print(
    "Same HSG group:",
    int(review["HSG_Group_Match"].sum())
)

print(
    "Positive candidate elevation overlap:",
    int(review["Positive_Elevation_Overlap"].sum())
)

print("\nREVIEW CATEGORIES")
display(
    review["Review_Category"].value_counts().to_frame(
        "Number_of_Candidates"
    )
)

print("\nLITHOLOGY COMPARISON")
display(comparison)

print("\nPAIR SUMMARY")
display(pair_summary)

print("\nWorkbook:", excel_file)
print("CSV:", csv_file)


**[orig #290]** Step 7G-1 — geological decision register (no automatic acceptance)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import os
import re
import pandas as pd
import numpy as np

BASE = (
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input"
)

INPUT = os.path.join(
    BASE, "Step7G_Geological_Assessment",
    "Step7G_Geological_Candidate_Register.csv"
)

OUT_DIR = os.path.join(
    BASE, "Step7G_Geological_Assessment"
)

df = pd.read_csv(INPUT)

def normalize(text):
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text).lower()).strip()

def lithology_family(text):
    t = normalize(text)

    if "quartzite" in t:
        return "QUARTZITE"

    if "shale" in t:
        return "SHALE"

    if "slate" in t:
        return "SLATE"

    if "phyllite" in t:
        return "PHYLLITE"

    if "schist" in t:
        return "SCHIST"

    if "kankar" in t or "sand" in t or "clay" in t or "silt" in t:
        return "GRANULAR_OR_FINE_MIXED_DEPOSIT"

    if "weathered" in t:
        return "WEATHERED_ROCK"

    return "OTHER_OR_UNRESOLVED"

def compare_lithology(row):

    a = normalize(row["A_Original_Lithology"])
    b = normalize(row["B_Original_Lithology"])

    family_a = lithology_family(a)
    family_b = lithology_family(b)

    if not a or not b:
        return pd.Series([
            family_a, family_b,
            "INSUFFICIENT_DESCRIPTION",
            "UNRESOLVED",
            "Original lithology description requires verification."
        ])

    if a == b:
        return pd.Series([
            family_a, family_b,
            "EXACT_DESCRIPTION_MATCH",
            "STRONG_DESCRIPTIVE_MATCH",
            "Descriptions are identical; geological continuity remains unverified."
        ])

    if family_a != family_b:
        return pd.Series([
            family_a, family_b,
            "DIFFERENT_LITHOLOGY_FAMILIES",
            "DO_NOT_CORRELATE_DIRECTLY",
            "Different source lithology families despite broad HSG agreement."
        ])

    if family_a in ["SHALE", "QUARTZITE", "SLATE", "PHYLLITE", "SCHIST"]:
        return pd.Series([
            family_a, family_b,
            "SAME_ROCK_FAMILY_DIFFERENT_DESCRIPTION",
            "POSSIBLE_MATCH_REQUIRES_REVIEW",
            "Same broad rock family but differences in description must be reviewed."
        ])

    return pd.Series([
        family_a, family_b,
        "RELATED_DEPOSIT_DESCRIPTION",
        "UNCERTAIN_CORRELATION",
        "Broad material family agrees but grain size or fines content differs."
    ])

df[
    [
        "Lithology_Family_A",
        "Lithology_Family_B",
        "Lithology_Comparison",
        "Preliminary_Interpretation",
        "Interpretation_Reason"
    ]
] = df.apply(compare_lithology, axis=1)

# No automatic geological acceptance.
df["Final_Geological_Correlation"] = "PENDING"
df["Aquifer_Aquitard_Status"] = "NOT_ASSIGNED"
df["MODFLOW_Layer"] = "NOT_ASSIGNED"
df["Reviewer_Approval"] = "REQUIRED"

output_csv = os.path.join(
    OUT_DIR, "Step7G1_Geological_Decision_Register.csv"
)

output_xlsx = os.path.join(
    OUT_DIR, "Step7G1_Geological_Decision_Register.xlsx"
)

summary = (
    df.groupby("Preliminary_Interpretation")
    .size()
    .reset_index(name="Candidate_Count")
)

with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    df.to_excel(
        writer, sheet_name="Geological_Decisions", index=False
    )
    summary.to_excel(
        writer, sheet_name="Interpretation_Summary", index=False
    )

df.to_csv(output_csv, index=False)

print("========== STEP 7G-1 COMPLETED ==========")
print("Candidate records:", len(df))
print("\nPreliminary interpretation:")
display(summary)

print("\nCandidate register:")
display(df[
    [
        "Borehole_A", "Borehole_B",
        "A_Original_Lithology",
        "B_Original_Lithology",
        "Lithology_Comparison",
        "Preliminary_Interpretation"
    ]
])

print("\nExcel:", output_xlsx)
print("CSV:", output_csv)


## 3. Preliminary sections (Step 7H)

**[orig #291]** **MAIN** Step 7H — preliminary elevation-based sections (four ATAL sections)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ================================================================
# STEP 7H — PRELIMINARY ELEVATION-BASED HYDROSTRATIGRAPHIC SECTIONS
#
# Four ATAL sections
# Source-preserving
# No automatic geological correlation
# No MODFLOW layer assignment
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

BASE = (
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input"
)

HSG_FILE = os.path.join(
    BASE, "Alwar_Hydrostratigraphic_Intervals.csv"
)

ELEV_FILE = os.path.join(
    BASE, "Step6_Elevation_Validation",
    "Step6F_Borehole_Lithology_Elevation_Database.csv"
)

COORD_FILE = os.path.join(
    BASE, "Step6_Elevation_Validation",
    "Step6B_Borehole_Coordinates_For_DEM.csv"
)

OUT_DIR = os.path.join(
    BASE, "Step7H_Preliminary_Cross_Sections"
)

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------
# 2. SECTION DEFINITIONS
# ------------------------------------------------

sections = {
    "Section_A": [
        "ATAL/0522/ALW/RAJ/04",
        "ATAL/0522/ALW/RAJ/05",
        "ATAL/0522/ALW/RAJ/06",
        "ATAL/0522/ALW/RAJ/14"
    ],

    "Section_B": [
        "ATAL/0522/ALW/RAJ/05",
        "ATAL/0522/ALW/RAJ/13"
    ],

    "Section_C": [
        "ATAL/0522/ALW/RAJ/10",
        "ATAL/0522/ALW/RAJ/15"
    ],

    "Section_D": [
        "ATAL/0522/ALW/RAJ/12",
        "ATAL/0522/ALW/RAJ/16"
    ]
}

# ------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------

hsg = pd.read_csv(HSG_FILE)
elev = pd.read_csv(ELEV_FILE)
coords = pd.read_csv(COORD_FILE)

print("HSG interval records:", len(hsg))
print("Elevation records:", len(elev))
print("Coordinate records:", len(coords))

# ------------------------------------------------
# 4. VALIDATE REQUIRED COLUMNS
# ------------------------------------------------

required_hsg = [
    "Borehole_ID",
    "Source_From_m",
    "Source_To_m",
    "Hydrostratigraphic_Group_ID",
    "Source_Lithology"
]

required_elev = [
    "Borehole_ID",
    "Ground_Elevation_Candidate_m",
    "Elevation_Source_Selected",
    "Elevation_Confidence_Status"
]

required_coords = [
    "Borehole_ID",
    "Latitude",
    "Longitude"
]

for col in required_hsg:
    if col not in hsg.columns:
        raise ValueError(f"HSG file missing column: {col}")

for col in required_elev:
    if col not in elev.columns:
        raise ValueError(f"Elevation file missing column: {col}")

for col in required_coords:
    if col not in coords.columns:
        raise ValueError(f"Coordinate file missing column: {col}")

# ------------------------------------------------
# 5. NUMERIC CONVERSION
# ------------------------------------------------

for col in ["Source_From_m", "Source_To_m"]:
    hsg[col] = pd.to_numeric(hsg[col], errors="coerce")

for col in [
    "Ground_Elevation_Candidate_m"
]:
    elev[col] = pd.to_numeric(elev[col], errors="coerce")

for col in ["Latitude", "Longitude"]:
    coords[col] = pd.to_numeric(coords[col], errors="coerce")

# ------------------------------------------------
# 6. ONE ELEVATION AND COORDINATE RECORD PER BOREHOLE
# ------------------------------------------------

elev_fields = [
    "Borehole_ID",
    "Ground_Elevation_Candidate_m",
    "Elevation_Source_Selected",
    "Elevation_Confidence_Status"
]

elev_unique = elev[elev_fields].drop_duplicates()

if elev_unique["Borehole_ID"].duplicated().any():
    raise ValueError(
        "Conflicting elevation records found for one or more boreholes. "
        "Inspect Step 6F before continuing."
    )

coords_unique = coords[required_coords].drop_duplicates()

if coords_unique["Borehole_ID"].duplicated().any():
    raise ValueError(
        "Conflicting coordinate records found. "
        "Inspect Step 6B before continuing."
    )

# ------------------------------------------------
# 7. JOIN HSG, ELEVATION AND COORDINATES
# ------------------------------------------------

data = hsg.merge(
    elev_unique,
    on="Borehole_ID",
    how="left",
    validate="many_to_one"
)

data = data.merge(
    coords_unique,
    on="Borehole_ID",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------
# 8. FILTER THE 13 ATAL BOREHOLES
# ------------------------------------------------

all_section_wells = sorted(
    set(w for group in sections.values() for w in group)
)

data = data[data["Borehole_ID"].isin(all_section_wells)].copy()

missing_elev = data[
    data["Ground_Elevation_Candidate_m"].isna()
]["Borehole_ID"].unique()

missing_coord = data[
    data["Latitude"].isna() | data["Longitude"].isna()
]["Borehole_ID"].unique()

if len(missing_elev) > 0:
    raise ValueError(
        f"Missing ground elevation for: {missing_elev.tolist()}"
    )

if len(missing_coord) > 0:
    raise ValueError(
        f"Missing coordinates for: {missing_coord.tolist()}"
    )

# ------------------------------------------------
# 9. CALCULATE PROVISIONAL INTERVAL ELEVATIONS
# ------------------------------------------------

# Depth is measured below ground level.
# Z = ground elevation candidate - depth below ground.

data["Candidate_Z_Top_m"] = (
    data["Ground_Elevation_Candidate_m"] -
    data["Source_From_m"]
)

data["Candidate_Z_Bottom_m"] = (
    data["Ground_Elevation_Candidate_m"] -
    data["Source_To_m"]
)

data["Interval_Thickness_m"] = (
    data["Source_To_m"] - data["Source_From_m"]
)

# ------------------------------------------------
# 10. DEFINE HSG COLOURS
# ------------------------------------------------

hsg_colors = {
    "HSG-1": "#E6C65B",
    "HSG-2": "#75AADB",
    "HSG-3": "#6FAF83",
    "HSG-4": "#A58BC2",
    "HSG-5": "#C7A27C"
}

# ------------------------------------------------
# 11. CALCULATE DISTANCE ALONG EACH SECTION
# ------------------------------------------------

def section_positions(well_ids):
    """
    Calculate relative distance along a straight section axis.

    For two wells, the axis follows A to B.
    For four wells, PCA determines the dominant spatial axis.
    Distances are approximate projected distances in km.
    """

    subset = (
        coords_unique[
            coords_unique["Borehole_ID"].isin(well_ids)
        ]
        .set_index("Borehole_ID")
        .loc[well_ids]
    )

    lat = subset["Latitude"].to_numpy()
    lon = subset["Longitude"].to_numpy()

    lat0 = np.mean(lat)

    x = (
        (lon - np.mean(lon)) *
        111.32 * np.cos(np.radians(lat0))
    )

    y = (
        (lat - np.mean(lat)) * 110.57
    )

    xy = np.column_stack([x, y])

    if len(well_ids) == 2:
        direction = xy[1] - xy[0]
        direction = direction / np.linalg.norm(direction)
    else:
        centered = xy - xy.mean(axis=0)
        _, _, vt = np.linalg.svd(
            centered, full_matrices=False
        )
        direction = vt[0]

    projected = xy @ direction

    # Fix direction for reproducible plots.
    if projected[-1] < projected[0]:
        projected = -projected

    projected = projected - projected.min()

    return dict(zip(well_ids, projected))

# ------------------------------------------------
# 12. PLOT ONE CROSS-SECTION
# ------------------------------------------------

def plot_section(section_name, well_ids, pdf=None):

    positions = section_positions(well_ids)

    fig, ax = plt.subplots(figsize=(13, 7))

    section_data = data[
        data["Borehole_ID"].isin(well_ids)
    ].copy()

    # Plot each borehole independently.
    for well in well_ids:

        w = section_data[
            section_data["Borehole_ID"] == well
        ].sort_values("Source_From_m")

        x = positions[well]

        ground = w["Ground_Elevation_Candidate_m"].iloc[0]

        # Draw ground elevation marker.
        ax.scatter(
            x, ground,
            marker="^",
            s=65,
            color="black",
            zorder=5
        )

        ax.text(
            x,
            ground + 7,
            well.split("/")[-1],
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold"
        )

        # Draw individual lithology/HSG intervals.
        for _, r in w.iterrows():

            top = r["Candidate_Z_Top_m"]
            bottom = r["Candidate_Z_Bottom_m"]
            height = top - bottom

            group = r["Hydrostratigraphic_Group_ID"]
            color = hsg_colors.get(group, "lightgrey")

            rect = Rectangle(
                (x - 0.32, bottom),
                0.64,
                height,
                facecolor=color,
                edgecolor="black",
                linewidth=0.7,
                alpha=0.9
            )

            ax.add_patch(rect)

            # Label sufficiently thick intervals.
            if height >= 8:
                ax.text(
                    x,
                    (top + bottom) / 2,
                    group,
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="black"
                )

    # Ground elevation range.
    ground_values = (
        section_data.groupby("Borehole_ID")
        ["Ground_Elevation_Candidate_m"].first()
    )

    min_z = section_data["Candidate_Z_Bottom_m"].min()
    max_z = section_data["Ground_Elevation_Candidate_m"].max()

    ax.set_ylim(
        min_z - 10,
        max_z + 25
    )

    ax.set_xlim(
        -0.8,
        max(positions.values()) + 0.8
    )

    ax.set_xticks(
        [positions[w] for w in well_ids]
    )

    ax.set_xticklabels(
        [w.split("/")[-1] for w in well_ids]
    )

    ax.set_xlabel(
        "Relative projected distance along section (km)"
    )

    ax.set_ylabel(
        "Candidate elevation (m; datum unverified)"
    )

    ax.set_title(
        f"{section_name}: Preliminary Hydrostratigraphic Section\n"
        "ATAL boreholes — provisional elevation-based display"
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.4
    )

    legend_items = [
        Rectangle(
            (0, 0), 1, 1,
            facecolor=color,
            edgecolor="black",
            label=group
        )
        for group, color in hsg_colors.items()
    ]

    legend_items.append(
        Line2D(
            [0], [0],
            marker="^",
            color="black",
            linestyle="None",
            label="Ground elevation candidate"
        )
    )

    ax.legend(
        handles=legend_items,
        loc="upper right",
        fontsize=8,
        ncol=2
    )

    fig.text(
        0.5, 0.015,
        "No geological boundaries are correlated or connected. "
        "Reported ATAL ground elevations have unverified vertical datum. "
        "HSG groups are descriptive lithological groupings, not confirmed "
        "aquifers/aquitards.",
        ha="center",
        fontsize=8,
        wrap=True
    )

    plt.tight_layout(rect=[0, 0.06, 1, 1])

    png_file = os.path.join(
        OUT_DIR,
        f"Step7H_{section_name}.png"
    )

    fig.savefig(
        png_file,
        dpi=300,
        bbox_inches="tight"
    )

    if pdf is not None:
        pdf.savefig(
            fig,
            bbox_inches="tight"
        )

    plt.show()
    plt.close(fig)

    return positions

# ------------------------------------------------
# 13. CREATE FOUR SECTIONS
# ------------------------------------------------

positions_records = []

pdf_file = os.path.join(
    OUT_DIR,
    "Step7H_Four_Preliminary_Hydrostratigraphic_Sections.pdf"
)

with PdfPages(pdf_file) as pdf:

    for section_name, wells in sections.items():

        print("\nCreating:", section_name)

        positions = plot_section(
            section_name,
            wells,
            pdf=pdf
        )

        for well, position in positions.items():
            positions_records.append({
                "Section": section_name,
                "Borehole_ID": well,
                "Relative_Projected_Distance_km": position
            })

# ------------------------------------------------
# 14. EXPORT SECTION INTERVAL DATABASE
# ------------------------------------------------

output_cols = [
    "Borehole_ID",
    "Source_From_m",
    "Source_To_m",
    "Interval_Thickness_m",
    "Source_Lithology",
    "Hydrostratigraphic_Group_ID",
    "Ground_Elevation_Candidate_m",
    "Candidate_Z_Top_m",
    "Candidate_Z_Bottom_m",
    "Elevation_Source_Selected",
    "Elevation_Confidence_Status",
    "Latitude",
    "Longitude"
]

section_intervals = data[output_cols].copy()

interval_file = os.path.join(
    OUT_DIR,
    "Step7H_ATAL_Cross_Section_Intervals.csv"
)

section_intervals.to_csv(
    interval_file,
    index=False
)

positions_file = os.path.join(
    OUT_DIR,
    "Step7H_Section_Borehole_Positions.csv"
)

pd.DataFrame(positions_records).to_csv(
    positions_file,
    index=False
)

# ------------------------------------------------
# 15. EXPORT ELEVATION SUMMARY
# ------------------------------------------------

elevation_summary = (
    data.groupby("Borehole_ID")
    .agg(
        Ground_Elevation_Candidate_m=(
            "Ground_Elevation_Candidate_m", "first"
        ),
        Elevation_Source=(
            "Elevation_Source_Selected", "first"
        ),
        Elevation_Confidence=(
            "Elevation_Confidence_Status", "first"
        ),
        Maximum_Depth_m=(
            "Source_To_m", "max"
        ),
        Deepest_Interval_Elevation_m=(
            "Candidate_Z_Bottom_m", "min"
        )
    )
    .reset_index()
)

elevation_summary.to_csv(
    os.path.join(
        OUT_DIR,
        "Step7H_Borehole_Elevation_Summary.csv"
    ),
    index=False
)

# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n========== STEP 7H COMPLETED ==========")

print("Sections generated:", len(sections))
print("Boreholes included:", data["Borehole_ID"].nunique())
print("Lithology intervals plotted:", len(data))

print("\nSECTION BOREHOLE COUNTS")
for name, wells in sections.items():
    print(name, ":", len(wells))

print("\nPDF:", pdf_file)
print("Interval CSV:", interval_file)
print("Position CSV:", positions_file)
print("Output folder:", OUT_DIR)

print("\nELEVATION CONFIDENCE")
display(
    elevation_summary[
        ["Elevation_Source", "Elevation_Confidence"]
    ].value_counts().to_frame("Borehole_Count")
)

display(elevation_summary)


**[orig #292]** Step 7H-1 — cross-track distance of boreholes from each section (needs `coords` from the cell above)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
# Borehole membership based on the Step 7H sections
sections = {
    "Section_A": [
        "ATAL/0522/ALW/RAJ/04",
        "ATAL/0522/ALW/RAJ/05",
        "ATAL/0522/ALW/RAJ/06",
        "ATAL/0522/ALW/RAJ/14"
    ],
    "Section_B": [
        "ATAL/0522/ALW/RAJ/05",
        "ATAL/0522/ALW/RAJ/13"
    ],
    "Section_C": [
        "ATAL/0522/ALW/RAJ/10",
        "ATAL/0522/ALW/RAJ/15"
    ],
    "Section_D": [
        "ATAL/0522/ALW/RAJ/12",
        "ATAL/0522/ALW/RAJ/16"
    ]
}

# Check required columns
required = ["Borehole_ID", "Latitude", "Longitude"]
missing = [c for c in required if c not in coords.columns]

if missing:
    raise ValueError(f"Missing coordinate columns: {missing}")

coords["Borehole_ID"] = coords["Borehole_ID"].astype(str)

# Convert latitude/longitude to local coordinates in km
# Equirectangular approximation around the study-area centroid
lat0 = np.deg2rad(coords["Latitude"].mean())
R = 6371.0088  # Earth mean radius in km

coords["X_km"] = (
    R * np.deg2rad(coords["Longitude"]) * np.cos(lat0)
)
coords["Y_km"] = R * np.deg2rad(coords["Latitude"])

results = []
section_summaries = []

for section_name, borehole_ids in sections.items():

    subset = coords[
        coords["Borehole_ID"].isin(borehole_ids)
    ].copy()

    missing_ids = sorted(set(borehole_ids) - set(subset["Borehole_ID"]))

    if missing_ids:
        raise ValueError(
            f"{section_name}: missing boreholes in coordinate file: "
            f"{missing_ids}"
        )

    if subset["Borehole_ID"].duplicated().any():
        raise ValueError(
            f"{section_name}: duplicate borehole coordinate records found."
        )

    xy = subset[["X_km", "Y_km"]].to_numpy()
    centroid = xy.mean(axis=0)
    centered = xy - centroid

    if len(subset) == 2:
        # Transect is the straight line between the two boreholes
        direction = xy[1] - xy[0]
        direction = direction / np.linalg.norm(direction)
        method = "Two-point connecting line"

    else:
        # PCA best-fit axis through the boreholes
        _, _, vt = np.linalg.svd(centered, full_matrices=False)
        direction = vt[0]
        direction = direction / np.linalg.norm(direction)
        method = "PCA best-fit transect"

    # Perpendicular unit vector
    perpendicular = np.array([-direction[1], direction[0]])

    # Signed along-track and cross-track distances from centroid
    along = centered @ direction
    cross_signed = centered @ perpendicular
    cross_abs = np.abs(cross_signed)

    for i, (_, row) in enumerate(subset.iterrows()):
        results.append({
            "Section": section_name,
            "Borehole_ID": row["Borehole_ID"],
            "Latitude": row["Latitude"],
            "Longitude": row["Longitude"],
            "Transect_Method": method,
            "Along_Track_km": along[i],
            "Cross_Track_Signed_km": cross_signed[i],
            "Cross_Track_Absolute_km": cross_abs[i],
            "Cross_Track_Absolute_m": cross_abs[i] * 1000
        })

    section_summaries.append({
        "Section": section_name,
        "Number_of_Boreholes": len(subset),
        "Transect_Method": method,
        "Maximum_Cross_Track_km": cross_abs.max(),
        "Mean_Cross_Track_km": cross_abs.mean(),
        "Maximum_Cross_Track_m": cross_abs.max() * 1000,
        "Mean_Cross_Track_m": cross_abs.mean() * 1000
    })

cross_track = pd.DataFrame(results)
summary = pd.DataFrame(section_summaries)

# Sort boreholes along the calculated transect
cross_track = cross_track.sort_values(
    ["Section", "Along_Track_km"]
).reset_index(drop=True)

print("Cross-track calculations completed.")
display(cross_track.round(3))
display(summary.round(3))

**[orig #293]** Step 7H-1 — section-level interpretation; save (needs the cell above)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
def classify_cross_track(distance_m):
    if distance_m <= 500:
        return "LOW_OFFSET"
    elif distance_m <= 1000:
        return "MODERATE_OFFSET"
    elif distance_m <= 2000:
        return "HIGH_OFFSET"
    else:
        return "VERY_HIGH_OFFSET"

cross_track["Geometry_Screening"] = (
    cross_track["Cross_Track_Absolute_m"]
    .apply(classify_cross_track)
)

# Add section-level interpretation
summary["Geometry_Screening"] = summary[
    "Maximum_Cross_Track_m"
].apply(classify_cross_track)

# Save outputs
cross_track_file = os.path.join(
    OUT_DIR, "Step7H1_Borehole_Cross_Track_Audit.csv"
)

summary_file = os.path.join(
    OUT_DIR, "Step7H1_Section_Geometry_Summary.csv"
)

cross_track.to_csv(cross_track_file, index=False)
summary.to_csv(summary_file, index=False)

print("Saved:")
print(cross_track_file)
print(summary_file)

print("\nSection summary:")
display(summary.round(3))

print("\nBorehole-level cross-track distances:")
display(
    cross_track[
        [
            "Section",
            "Borehole_ID",
            "Along_Track_km",
            "Cross_Track_Absolute_m",
            "Geometry_Screening"
        ]
    ].round(2)
)

**[orig #294]** Step 7H-1 — plot cross-track geometry (needs the cells above)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for ax, (section_name, borehole_ids) in zip(axes, sections.items()):

    sub = cross_track[
        cross_track["Section"] == section_name
    ].copy()

    ax.scatter(
        sub["Along_Track_km"],
        sub["Cross_Track_Signed_km"],
        s=90
    )

    for _, row in sub.iterrows():
        ax.annotate(
            row["Borehole_ID"].split("/")[-1],
            (
                row["Along_Track_km"],
                row["Cross_Track_Signed_km"]
            ),
            xytext=(5, 5),
            textcoords="offset points"
        )

    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_title(section_name)
    ax.set_xlabel("Along-track distance (km)")
    ax.set_ylabel("Signed cross-track distance (km)")
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Step 7H-1: Borehole Cross-Track Geometry",
    fontsize=14
)
plt.tight_layout()

plot_file = os.path.join(
    OUT_DIR, "Step7H1_Cross_Track_Geometry.png"
)

plt.savefig(plot_file, dpi=300, bbox_inches="tight")
plt.show()

print("Plot saved:", plot_file)

**[orig #295]** Check the input files and columns for the revised sections  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import os
import pandas as pd

BASE = (
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input"
)

FILES = {
    "Hydrostratigraphic": os.path.join(
        BASE, "Alwar_Hydrostratigraphic_Intervals.csv"
    ),
    "Elevation": os.path.join(
        BASE, "Step6_Elevation_Validation",
        "Step6F_Borehole_Lithology_Elevation_Database.csv"
    ),
    "Coordinates": os.path.join(
        BASE, "Step6_Elevation_Validation",
        "Step6B_Borehole_Coordinates_For_DEM.csv"
    )
}

for name, path in FILES.items():
    print("\n" + "=" * 70)
    print(name)
    print("File exists:", os.path.exists(path))

    if not os.path.exists(path):
        continue

    df = pd.read_csv(path)

    print("Rows:", len(df))
    print("Columns:")
    for col in df.columns:
        print(" -", col)

    print("\nFirst two records:")
    display(df.head(2))

**[orig #296]** **MAIN** Step 7H-2 — revised five preliminary sections  
<sub>• output not saved in the source notebook</sub>

In [ ]:
# ============================================================
# STEP 7H-2: REVISED FIVE PRELIMINARY HYDROSTRATIGRAPHIC SECTIONS
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

BASE = (
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input"
)

HYDRO_FILE = os.path.join(
    BASE, "Alwar_Hydrostratigraphic_Intervals.csv"
)

ELEV_FILE = os.path.join(
    BASE,
    "Step6_Elevation_Validation",
    "Step6F_Borehole_Lithology_Elevation_Database.csv"
)

COORD_FILE = os.path.join(
    BASE,
    "Step6_Elevation_Validation",
    "Step6B_Borehole_Coordinates_For_DEM.csv"
)

OUT_DIR = os.path.join(
    BASE, "Step7H2_Revised_Five_Sections"
)

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. READ INPUTS
# ------------------------------------------------------------

hydro = pd.read_csv(HYDRO_FILE)
elev = pd.read_csv(ELEV_FILE)
coords = pd.read_csv(COORD_FILE)

print("Hydrostratigraphic records:", len(hydro))
print("Elevation records:", len(elev))
print("Coordinate records:", len(coords))

# ------------------------------------------------------------
# 3. VALIDATE REQUIRED COLUMNS
# ------------------------------------------------------------

hydro_required = [
    "Borehole_ID",
    "Source_Row",
    "Source_From_m",
    "Source_To_m",
    "Source_Lithology",
    "Hydrostratigraphic_Group_ID"
]

elev_required = [
    "Borehole_ID",
    "Source_Row",
    "Lithology_Top_Elevation_Candidate_m",
    "Lithology_Bottom_Elevation_Candidate_m",
    "Ground_Elevation_Candidate_m",
    "Elevation_Confidence_Status",
    "Elevation_Source_Selected"
]

coord_required = [
    "Borehole_ID",
    "Latitude",
    "Longitude",
    "Coordinate_Source"
]

for name, df, required in [
    ("Hydrostratigraphic", hydro, hydro_required),
    ("Elevation", elev, elev_required),
    ("Coordinates", coords, coord_required)
]:
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"{name} file is missing columns: {missing}"
        )

# Normalize matching keys without changing source values
for df in [hydro, elev, coords]:
    df["Borehole_ID"] = df["Borehole_ID"].astype(str)

hydro["Source_Row"] = hydro["Source_Row"].astype(str)
elev["Source_Row"] = elev["Source_Row"].astype(str)

# ------------------------------------------------------------
# 4. DEFINE FIVE REVISED SECTIONS
# ------------------------------------------------------------

sections = {
    "Section_A1": [
        "ATAL/0522/ALW/RAJ/04",
        "ATAL/0522/ALW/RAJ/05"
    ],
    "Section_A2": [
        "ATAL/0522/ALW/RAJ/14",
        "ATAL/0522/ALW/RAJ/06"
    ],
    "Section_B": [
        "ATAL/0522/ALW/RAJ/05",
        "ATAL/0522/ALW/RAJ/13"
    ],
    "Section_C": [
        "ATAL/0522/ALW/RAJ/10",
        "ATAL/0522/ALW/RAJ/15"
    ],
    "Section_D": [
        "ATAL/0522/ALW/RAJ/12",
        "ATAL/0522/ALW/RAJ/16"
    ]
}

selected_ids = sorted(set(
    bh for pair in sections.values() for bh in pair
))

print("\nUnique selected boreholes:", len(selected_ids))

# Confirm all selected boreholes have coordinates
missing_coords = sorted(
    set(selected_ids) - set(coords["Borehole_ID"])
)

if missing_coords:
    raise ValueError(
        f"Missing coordinates for: {missing_coords}"
    )

# ------------------------------------------------------------
# 5. VALIDATE SOURCE INTERVAL MATCHING
# ------------------------------------------------------------

if hydro.duplicated(["Borehole_ID", "Source_Row"]).any():
    raise ValueError(
        "Duplicate hydrostratigraphic Borehole_ID + Source_Row keys."
    )

if elev.duplicated(["Borehole_ID", "Source_Row"]).any():
    raise ValueError(
        "Duplicate elevation Borehole_ID + Source_Row keys."
    )

# Keep all source and classification columns
plot_data = hydro.merge(
    elev,
    on=["Borehole_ID", "Source_Row"],
    how="left",
    suffixes=("_Hydro", "_Elevation"),
    indicator=True,
    validate="one_to_one"
)

unmatched = plot_data[
    plot_data["_merge"] != "both"
].copy()

if len(unmatched) > 0:
    unmatched.to_csv(
        os.path.join(OUT_DIR, "Step7H2_Unmatched_Intervals.csv"),
        index=False
    )

    raise ValueError(
        f"{len(unmatched)} hydrostratigraphic intervals "
        "did not match the elevation database."
    )

plot_data.drop(columns="_merge", inplace=True)

# Filter the selected boreholes
plot_data = plot_data[
    plot_data["Borehole_ID"].isin(selected_ids)
].copy()

# Convert plotting fields to numeric
numeric_fields = [
    "Lithology_Top_Elevation_Candidate_m",
    "Lithology_Bottom_Elevation_Candidate_m",
    "Ground_Elevation_Candidate_m",
    "Source_From_m",
    "Source_To_m"
]

for col in numeric_fields:
    plot_data[col] = pd.to_numeric(
        plot_data[col], errors="coerce"
    )

# Exclude only records that cannot be plotted due to missing
# candidate elevation values. Retain them in the full audit.
plot_data["Plot_Elevation_Valid"] = (
    plot_data[
        [
            "Lithology_Top_Elevation_Candidate_m",
            "Lithology_Bottom_Elevation_Candidate_m"
        ]
    ].notna().all(axis=1)
)

print("\nMatched selected interval records:", len(plot_data))
print(
    "Records with valid candidate elevations:",
    plot_data["Plot_Elevation_Valid"].sum()
)

# ------------------------------------------------------------
# 6. COORDINATE AND DISTANCE CALCULATION
# ------------------------------------------------------------

coord_selected = coords[
    coords["Borehole_ID"].isin(selected_ids)
].copy()

if coord_selected["Borehole_ID"].duplicated().any():
    raise ValueError("Duplicate coordinate records detected.")

coord_lookup = coord_selected.set_index("Borehole_ID")

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088

    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))

position_records = []

for section_name, pair in sections.items():

    bh1, bh2 = pair

    p1 = coord_lookup.loc[bh1]
    p2 = coord_lookup.loc[bh2]

    distance = haversine_km(
        p1["Latitude"], p1["Longitude"],
        p2["Latitude"], p2["Longitude"]
    )

    position_records.extend([
        {
            "Section": section_name,
            "Borehole_ID": bh1,
            "Position_km": 0.0,
            "Latitude": p1["Latitude"],
            "Longitude": p1["Longitude"],
            "Coordinate_Source": p1["Coordinate_Source"]
        },
        {
            "Section": section_name,
            "Borehole_ID": bh2,
            "Position_km": distance,
            "Latitude": p2["Latitude"],
            "Longitude": p2["Longitude"],
            "Coordinate_Source": p2["Coordinate_Source"]
        }
    ])

    print(
        f"{section_name}: {bh1.split('/')[-1]} – "
        f"{bh2.split('/')[-1]} = {distance:.3f} km"
    )

positions = pd.DataFrame(position_records)

# ------------------------------------------------------------
# 7. HSG COLOUR DEFINITIONS
# ------------------------------------------------------------

hsg_colors = {
    "HSG-1": "#E8C75E",
    "HSG-2": "#78A6D8",
    "HSG-3": "#65A985",
    "HSG-4": "#A68BC2",
    "HSG-5": "#C99B68"
}

# ------------------------------------------------------------
# 8. BUILD SECTION PLOTTING DATABASE
# ------------------------------------------------------------

position_lookup = positions.set_index(
    ["Section", "Borehole_ID"]
)["Position_km"].to_dict()

section_plot_records = []

for section_name, pair in sections.items():

    for bh in pair:

        bh_data = plot_data[
            plot_data["Borehole_ID"] == bh
        ].copy()

        bh_data["Section"] = section_name
        bh_data["Position_km"] = position_lookup[
            (section_name, bh)
        ]

        section_plot_records.append(bh_data)

section_plot_data = pd.concat(
    section_plot_records,
    ignore_index=True
)

# ------------------------------------------------------------
# 9. SAVE REVISED DATA REGISTERS
# ------------------------------------------------------------

section_plot_data.to_csv(
    os.path.join(
        OUT_DIR,
        "Step7H2_Revised_Section_Interval_Database.csv"
    ),
    index=False
)

positions.to_csv(
    os.path.join(
        OUT_DIR,
        "Step7H2_Revised_Borehole_Position_Register.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# 10. CREATE SECTION FIGURES
# ------------------------------------------------------------

pdf_path = os.path.join(
    OUT_DIR,
    "Step7H2_Five_Revised_Preliminary_Sections.pdf"
)

plot_records = []

with PdfPages(pdf_path) as pdf:

    for section_name, pair in sections.items():

        fig, ax = plt.subplots(figsize=(12, 7))

        valid_section = section_plot_data[
            (section_plot_data["Section"] == section_name)
            & (section_plot_data["Plot_Elevation_Valid"])
        ].copy()

        section_positions = positions[
            positions["Section"] == section_name
        ]

        # Plot each borehole
        for _, pos in section_positions.iterrows():

            bh = pos["Borehole_ID"]
            x = pos["Position_km"]

            bh_data = valid_section[
                valid_section["Borehole_ID"] == bh
            ].copy()

            bh_data = bh_data.sort_values(
                "Source_From_m"
            )

            # Draw lithological/HSG intervals
            for _, row in bh_data.iterrows():

                top = row[
                    "Lithology_Top_Elevation_Candidate_m"
                ]

                bottom = row[
                    "Lithology_Bottom_Elevation_Candidate_m"
                ]

                hsg = str(row["Hydrostratigraphic_Group_ID"])

                if pd.isna(top) or pd.isna(bottom):
                    continue

                y_bottom = min(top, bottom)
                height = abs(top - bottom)

                color = hsg_colors.get(hsg, "#BDBDBD")

                rect = Rectangle(
                    (x - 0.55, y_bottom),
                    1.10,
                    height,
                    facecolor=color,
                    edgecolor="black",
                    linewidth=0.7,
                    alpha=0.85
                )

                ax.add_patch(rect)

                # Label only if interval is tall enough
                if height >= 8:
                    ax.text(
                        x,
                        y_bottom + height / 2,
                        hsg,
                        ha="center",
                        va="center",
                        fontsize=7
                    )

                plot_records.append({
                    "Section": section_name,
                    "Borehole_ID": bh,
                    "Source_Row": row["Source_Row"],
                    "Source_From_m": row["Source_From_m"],
                    "Source_To_m": row["Source_To_m"],
                    "Source_Lithology": row["Source_Lithology"],
                    "HSG": hsg,
                    "Top_Elevation_m": top,
                    "Bottom_Elevation_m": bottom,
                    "Position_km": x
                })

            # Ground elevation candidate marker
            ground_values = pd.to_numeric(
                bh_data["Ground_Elevation_Candidate_m"],
                errors="coerce"
            ).dropna()

            if len(ground_values) > 0:

                ground = ground_values.iloc[0]

                ax.scatter(
                    x,
                    ground,
                    marker="^",
                    s=85,
                    color="black",
                    zorder=5,
                    label="Ground elevation candidate"
                    if bh == pair[0] else None
                )

                ax.annotate(
                    bh.split("/")[-1],
                    (x, ground),
                    xytext=(0, 8),
                    textcoords="offset points",
                    ha="center",
                    fontsize=10,
                    fontweight="bold"
                )

        # Axis formatting
        ax.set_xlim(-1.0, max(
            positions[
                positions["Section"] == section_name
            ]["Position_km"]
        ) + 1.0)

        all_y = valid_section[
            [
                "Lithology_Top_Elevation_Candidate_m",
                "Lithology_Bottom_Elevation_Candidate_m",
                "Ground_Elevation_Candidate_m"
            ]
        ].to_numpy(dtype=float)

        finite_y = all_y[np.isfinite(all_y)]

        if len(finite_y) > 0:
            ymin = np.min(finite_y)
            ymax = np.max(finite_y)
            margin = max(10, (ymax - ymin) * 0.08)

            ax.set_ylim(
                ymin - margin,
                ymax + margin
            )

        ax.set_xticks(
            section_positions["Position_km"]
        )

        ax.set_xticklabels([
            x.split("/")[-1]
            for x in section_positions["Borehole_ID"]
        ])

        ax.set_xlabel(
            "Distance along pair transect (km)"
        )

        ax.set_ylabel(
            "Candidate elevation (m; datum unverified)"
        )

        ax.set_title(
            f"{section_name}: Preliminary Hydrostratigraphic Section\n"
            "ATAL boreholes – provisional elevation-based display"
        )

        # Legend for HSG groups
        legend_handles = []

        for hsg, color in hsg_colors.items():

            legend_handles.append(
                Rectangle(
                    (0, 0), 1, 1,
                    facecolor=color,
                    edgecolor="black",
                    label=hsg
                )
            )

        ground_handle = plt.Line2D(
            [0], [0],
            marker="^",
            color="black",
            linestyle="None",
            label="Ground elevation candidate"
        )

        legend_handles.append(ground_handle)

        ax.legend(
            handles=legend_handles,
            loc="upper right",
            fontsize=8,
            ncol=3
        )

        ax.grid(axis="y", alpha=0.25)

        fig.text(
            0.5,
            0.015,
            "No geological boundaries are correlated or connected. "
            "Reported ATAL ground elevations have unverified vertical datum. "
            "HSG groups are descriptive lithological groupings, "
            "not confirmed aquifers/aquitards.",
            ha="center",
            fontsize=8
        )

        plt.tight_layout(rect=[0, 0.05, 1, 1])

        # Save individual PNG
        png_path = os.path.join(
            OUT_DIR,
            f"Step7H2_{section_name}.png"
        )

        fig.savefig(
            png_path,
            dpi=300,
            bbox_inches="tight"
        )

        # Add to combined PDF
        pdf.savefig(fig, bbox_inches="tight")

        plt.show()
        plt.close(fig)

# ------------------------------------------------------------
# 11. SAVE PLOTTED INTERVAL REGISTER
# ------------------------------------------------------------

plotted_intervals = pd.DataFrame(plot_records)

plotted_intervals.to_csv(
    os.path.join(
        OUT_DIR,
        "Step7H2_Plotted_Interval_Register.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# 12. CREATE VALIDATION SUMMARY
# ------------------------------------------------------------

validation_records = []

for section_name, pair in sections.items():

    section_data = section_plot_data[
        section_plot_data["Section"] == section_name
    ]

    valid = section_data[
        section_data["Plot_Elevation_Valid"]
    ]

    for bh in pair:

        bh_data = section_data[
            section_data["Borehole_ID"] == bh
        ]

        valid_bh = valid[
            valid["Borehole_ID"] == bh
        ]

        validation_records.append({
            "Section": section_name,
            "Borehole_ID": bh,
            "Input_Intervals": len(bh_data),
            "Plottable_Intervals": len(valid_bh),
            "Missing_Elevation_Intervals": (
                len(bh_data) - len(valid_bh)
            ),
            "Elevation_Confidence": (
                "; ".join(
                    bh_data["Elevation_Confidence_Status"]
                    .dropna().astype(str).unique()
                )
            ),
            "Elevation_Source": (
                "; ".join(
                    bh_data["Elevation_Source_Selected"]
                    .dropna().astype(str).unique()
                )
            )
        })

validation = pd.DataFrame(validation_records)

validation.to_csv(
    os.path.join(
        OUT_DIR,
        "Step7H2_Validation_Summary.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# 13. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 7H-2 COMPLETED")
print("=" * 65)

print("Output folder:", OUT_DIR)
print("Combined PDF:", pdf_path)

print("\nUnique boreholes:", len(selected_ids))
print("Revised sections:", len(sections))
print("Matched interval records:", len(section_plot_data))
print("Plotted interval records:", len(plotted_intervals))

print("\nValidation summary:")
display(validation)

print("\nSection distances:")
display(
    positions.groupby("Section")["Position_km"]
    .max()
    .reset_index(name="Pair_Distance_km")
)

print("\nGenerated files:")
for f in sorted(os.listdir(OUT_DIR)):
    print(" -", f)

**[orig #297]** Step 7H-2 fix — source-preserving interval merge (needs the cell above)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
# ============================================================
# STEP 7H-2 FIX: SOURCE-PRESERVING INTERVAL MERGE
# ============================================================

# Keep hydrostratigraphic source columns as the primary values.
# Import only elevation-specific fields from the elevation database.

elevation_fields = [
    "Borehole_ID",
    "Source_Row",
    "Lithology_Top_Elevation_Candidate_m",
    "Lithology_Bottom_Elevation_Candidate_m",
    "Ground_Elevation_Candidate_m",
    "Elevation_Confidence_Status",
    "Elevation_Source_Selected"
]

# Check duplicate keys
if hydro.duplicated(["Borehole_ID", "Source_Row"]).any():
    raise ValueError("Duplicate keys in hydrostratigraphic data.")

if elev.duplicated(["Borehole_ID", "Source_Row"]).any():
    raise ValueError("Duplicate keys in elevation data.")

# Merge without duplicating source lithology/depth columns
plot_data = hydro.merge(
    elev[elevation_fields],
    on=["Borehole_ID", "Source_Row"],
    how="left",
    validate="one_to_one",
    indicator=True
)

# Verify all interval matches
unmatched = plot_data[
    plot_data["_merge"] != "both"
].copy()

if len(unmatched) > 0:
    unmatched.to_csv(
        os.path.join(
            OUT_DIR,
            "Step7H2_Unmatched_Intervals.csv"
        ),
        index=False
    )

    raise ValueError(
        f"{len(unmatched)} intervals failed elevation matching."
    )

plot_data.drop(columns="_merge", inplace=True)

# Retain selected boreholes
plot_data = plot_data[
    plot_data["Borehole_ID"].isin(selected_ids)
].copy()

# Numeric conversion
numeric_fields = [
    "Lithology_Top_Elevation_Candidate_m",
    "Lithology_Bottom_Elevation_Candidate_m",
    "Ground_Elevation_Candidate_m",
    "Source_From_m",
    "Source_To_m"
]

for col in numeric_fields:
    plot_data[col] = pd.to_numeric(
        plot_data[col],
        errors="coerce"
    )

plot_data["Plot_Elevation_Valid"] = (
    plot_data[
        [
            "Lithology_Top_Elevation_Candidate_m",
            "Lithology_Bottom_Elevation_Candidate_m"
        ]
    ].notna().all(axis=1)
)

print("Corrected merge successful.")
print("Selected interval records:", len(plot_data))
print(
    "Intervals with valid elevations:",
    int(plot_data["Plot_Elevation_Valid"].sum())
)

print("\nSource columns preserved:")
print([
    "Source_From_m",
    "Source_To_m",
    "Source_Lithology",
    "Hydrostratigraphic_Group_ID"
])

display(plot_data.head())